# Intro

Check the readme files in the folder to learn how to run scans and use defaults vs specify parameters

# Initialize (Run everything in this section before starting experiments)

In [ ]:
## please keep the old lines and add new lines with minimal comments, so we can easily identify the scope of each test.
cfg_file='2026_05_01_smpd_v2_run28.yml'
expt_path = 'C:\\_Data\\SMPD\\2026_05_01_v2_run28_JPA'

max_t1 = 250 #

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import copy

#np.set_printoptions(legacy="1.25")
from qick import QickConfig

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
import slab_qick_calib.experiments as meas
from slab_qick_calib.calib import qubit_tuning, measure_func
from slab_qick_calib.calib.time_tracking import time_tracking
from slab_qick_calib.analysis import qubit_params, fitres
from slab_qick_calib.helpers import qick_check, config, handy
from fridge.bluefors import BlueforsClient
from fridge.credentials import BLUEFORS_HOST, BLUEFORS_PORT, BLUEFORS_API_KEY

%load_ext autoreload
%autoreload 2

# Set color palette and font size
handy.config_figs()
%config InlineBackend.figure_format = 'png'

import matplotlib 
matplotlib.rcParams['path.simplify'] = True
matplotlib.rcParams['path.simplify_threshold'] = 0.1
matplotlib.rcParams['agg.path.chunksize'] = 10000

## Set up new config 
Set to variables to True when setting up a new experiment config file. 

Note: make sure you set your ADC/DAC channels correctly. This code does not automatically fill in the ADC/DAC into your configuration file, so you should check yourself to make sure these values are correct. 

There are several elements that you may want to customize based on your readout parameters and coherence times. Check readme file config_manual.md

In [ ]:
# Set to false if you aren't creating a new one (so set to false as soon as you run it)
new_config = False
new_folder = True

nqubits = 2 # For SMPD, we have 1 qubit per chip, but it's coupled to 2 resonators, so we can treat it as a 2 qubit system for the purposes of the config file.
rfsoc_alias = 'smpd_qick'
t1_guess = 20 
ip = '192.168.137.93' # ip address of name server that rfsoc is connected to 
import os

configs_dir = os.path.join(os.getcwd(),'../../', 'configs')

cfg_file_path = os.path.join(configs_dir, cfg_file)
images_dir = os.path.join(expt_path, 'images')
data_dir = os.path.join(expt_path, 'data')
summary_dir = os.path.join(images_dir, 'summary')

if new_config or new_folder:
    if new_config:
        config.init_config(cfg_file_path, nqubits, type='full', aliases=rfsoc_alias, t1=t1_guess, ip=ip)
        config.init_model_config(cfg_file_path, nqubits)

    if not os.path.exists(expt_path):
        os.makedirs(expt_path)
        os.mkdir(images_dir)
        os.mkdir(summary_dir)
        os.mkdir(data_dir)

print('Data will be stored in', expt_path)

## Connect to RFSoC
Before running first cell, make sure a nameserver is running on the network, the Qick board is connected to it, and the ip address listed below matches that of the nameserver. 

You just need to run the first cell, then should be able to run any other cell in whatever order. 

If you need to restart the RFSoC, you should reconnect it to the nameserver and rerun this. 

In [ ]:
# Results config file
cfg_path = os.path.join(os.getcwd(),'../..', 'configs', cfg_file)
auto_cfg = config.load(cfg_path)

# print(auto_cfg)

# Connect to instruments
im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'], port=9090)
print(im)
soc = QickConfig(im[auto_cfg['aliases']['soc']].get_cfg())
print(soc)

cfg_dict = {'soc': soc, 'expt_path': expt_path, 'cfg_file': cfg_path, 'im': im}

In [ ]:
bf = BlueforsClient(
    host=BLUEFORS_HOST,
    port=BLUEFORS_PORT,
    api_key=BLUEFORS_API_KEY,
    verify_ssl=False,
)
print(f"MXC: {bf.get_mxc_temperature()*1000:.2f} mK")

# Applets (Not ncessary, run if needed)

## How to update config (you can also just edit yml directly)

In [ ]:
# #                                          param   value qubit #
# auto_cfg = config.update_readout(cfg_path, 'lamb', 5, qi)
# auto_cfg = config.update_qubit(cfg_path, 'f_ge', 5700, qi)

# # For multiple levels of nesting: 
# auto_cfg = config.update_qubit(cfg_path, ('pulses','pi_ge','gain'), 0.2, qi)

## Print scan params

In [ ]:
t1 = meas.T1Experiment(cfg_dict, qi=0, print=True)

In general, all scans will be interacted with either by running default, or giving arguments from params dict. 
You can run scans on list of different qubits or just one by adding first couple lines of each cell. 
Flag of update is used to decide if to set new config vals based on output of scan (if the fit looks good)

## Check QICK issues

### Check mirror frequencies on qubit

In [ ]:
qick_check.check_freqs(0, cfg_dict)

### Check mirror frequencies from resonators

In [ ]:
qick_check.check_resonances(cfg_dict)

### Check sampling rates and minimum point spacing

In [ ]:

fnyq = cfg_dict['soc']._get_ch_cfg(ro_ch=0)['f_dds']/2
clock_tick = 1e3*cfg_dict['soc'].cycles2us(1)
print(f'ADC Nyquist frequency is {fnyq} MHz')
print(f'1 clock tick is {clock_tick} ns')

### Make sure you're not near the nyquist frequency of the ADC

In [ ]:
qick_check.check_adc(cfg_dict)

# Time of Flight (TOF)

TOF measures the time it takes for the signal to run through the wires. It will give us the time in clock ticks that we should wait to make a measurements 

 Use this to set trig_offset in config file

In [ ]:
qi = 1
tof=meas.ToFCalibrationExperiment(cfg_dict=cfg_dict, qi=qi, params={'frequency':5000,'rounds':1000,'gain':1})

# Set frequency of choice and readout length (up to 13 us for standard ZCU216 firmware, readout)
#tof=meas.ToFCalibrationExperiment(cfg_dict=cfg_dict, qi=qi,params={'readout_length':13})#,)
        

## Set trig_offset to point where signal has appeared, usually around 300-500 ns

In [ ]:
tof_data = tof.analyze()
print(tof_data['xpts'][np.argmin(abs(tof_data['amps']-max(tof_data['amps'])/2))])

In [ ]:
qubit_list=[1]
trig_offset = tof_data['xpts'][np.argmin(abs(tof_data['amps']-max(tof_data['amps'])/2))]
for qi in qubit_list: 
    auto_cfg = config.update_readout(cfg_path, 'trig_offset', trig_offset, qi)

## Resonator ring up

In [ ]:
### anaylsis code for ring-up trajectory, to extract resonator parameters like Q and detuning. This is a more complex model than a simple exponential, as it accounts for the fact that the trajectory in the IQ plane can be a spiral due to detuning and phase effects.

def iq_ringup_model(t, i_ss, q_ss, tau, df, phase_offset):
    """
    Model for the complex IQ ring-up trajectory.
    Returns a flattened array of [I_data, Q_data].
    """
    # Complex steady state A
    A = i_ss + 1j * q_ss
    
    # Exponential rise with detuning rotation
    # Assuming start at origin (0,0)
    S = A * (1 - np.exp(-t / tau) * np.exp(1j * (2 * np.pi * df * t + phase_offset)))
    
    return np.concatenate([S.real, S.imag])

mask = (tof.data['xpts'] > auto_cfg.device.readout.trig_offset[qi])
t_data = tof.data['xpts'][mask]
ydata = tof.data['amps'][mask]
i_data = tof.data['i'][mask]
q_data = tof.data['q'][mask]

cdata = np.concatenate([i_data, q_data])
p0 = [i_data[-1], q_data[-1], (t_data[-1]-t_data[0])/5, 0, 0]
popt, pcov = curve_fit(iq_ringup_model, t_data, cdata, p0=p0)
print("Fit parameters:", popt)

i_ss, q_ss, tau, df, phase = popt
A = i_ss + 1j * q_ss
S_fit = A * (1 - np.exp(-t_data / tau) * np.exp(1j * (2 * np.pi * df * t_data + phase)))
i_fit = S_fit.real
q_fit = S_fit.imag

plt.figure()
plt.plot(t_data, ydata, 'o', label='Amp Data')
plt.plot(t_data, np.abs(i_fit + 1j * q_fit), '-', label='tau = {:.2f} us'.format(tau))
plt.xlabel('Time (us)')
plt.ylabel('Amplitude')
plt.legend()
plt.title(f'Resonator ring up for qi{qi}')
plt.tight_layout()
plt.savefig(os.path.join(summary_dir, f'resonator_ringup_qi{qi}.png'))
plt.show()
    
print(tau)
print(df)

## Once readout tuned up, check ring up and ring down.

In [ ]:
qi=1
avgs=3e4
readout_length=12
tof=meas.ToFCalibrationExperiment(cfg_dict=cfg_dict, qi=qi,params={'use_readout':True, 'rounds':avgs, 'readout_length':readout_length})#,params={'frequency':fi})
tof_e=meas.ToFCalibrationExperiment(cfg_dict=cfg_dict, qi=qi,params={'use_readout':True, 'rounds':avgs,'check_e':True, 'readout_length':readout_length})#,params={'frequency':fi})

# Resonator Spectroscopy 

Run resonator spectroscopy for all resonators by choosing a large frequency scan to look over. The scan will then find the different resonators and fill in the config file with their respective frequencies. In the autocalibration, there will be a finer sweep of each resonator to more accurately find its frequency. The frequencies are saved in <code>auto_cfg.device.readout.frequency</code>

## Coarse 

This will perform peak finding
Use params to specify frequency range, averaging, gain, expts. 
If gain is too high, you may be in punch out region, where resonators disappear, or above it 

In [ ]:
qi=1 # We only run this once for all qubits on a single feedline, this is a dummy value to make the scan work 
params={'start':7700, 'span':30, 'reps':500, 'gain': 0.01, 'expts':1000}
rspecc = meas.ResSpec(cfg_dict, qi=qi, style='coarse', progress=True, params=params)
res_values = rspecc.data['coarse_peaks']

Change prom (prominence value) to adjust how many peaks you find

In [ ]:
unwrapped = np.unwrap(rspecc.data['phases'])
polyfit = np.polyfit(rspecc.data['freq'][:200], unwrapped[:200], 1)
unwrapped -= np.polyval(polyfit, rspecc.data['freq'])

plt.plot(rspecc.data['freq'],unwrapped)

plt.figure()
IQ = rspecc.data['amps'] * np.exp(1j * unwrapped)
plt.plot(IQ.real, IQ.imag, 'o-')

Can delete values from res_values if they don't seem to be real res_values. 

## Fine

This will fit the resonance amplitude

Can run scan with default options, or specify your own, by commenting out different lines and editing paramters. 

### Fine resonator scan

Once the correct frequencies are saved to your config file. 

In [ ]:
auto_cfg = config.load(cfg_path)
update=False # Set to true if you want to update the config file with the new resonance values

# comment out one of these 
# qubit_list = np.arange(6)
qi = 1
ifauto = False

if ifauto:
    # Fully automated, using previous fit to kappa to set span
    rspec = meas.ResSpec(cfg_dict, qi=qi, params={'span':15,'reps':5000})
    gain = auto_cfg.device.readout.gain[qi]
else:
    # Manually set the span and gain 
    gain = 0.009
    rspec = meas.ResSpec(cfg_dict, qi=qi, params={'span':13, 'gain':gain, 'reps':1000, 'expts':2000})

if update: rspec.update()

In [ ]:
import h5py

# load most recent resonator spectroscopy data for qi1, file starts with resonator_spectroscopy_fine_qubit1
files = [f for f in os.listdir(expt_path +'\\data') if f.startswith('resonator_spectroscopy_fine_qubit1')]
files = sorted(files)
with h5py.File(expt_path +'\\data\\' + files[-1], 'r') as f:
    xpts = f['xpts'][:]
    amps = f['amps'][:]
    phases = f['phases'][:]
unwrapped = np.unwrap(phases)
polyfit = np.polyfit(xpts[:300],unwrapped[:300],1)
corrected = unwrapped - np.polyval(polyfit, xpts)
plt.plot(xpts, corrected)
plt.plot(xpts[:300], corrected[:300], 'r-')

S21 = amps * np.exp(1j * corrected)
start_idx = 0
end_idx = -1
plt.figure() 
plt.plot(S21.real[:], S21.imag[:])
plt.axis('equal')
fr, Qr, Qc_hat, a, phi, tau, Qc, _ , _ = fitres.finefit(xpts[start_idx:end_idx],S21[start_idx:end_idx],7740,np.array([0,0,0]))
fit_plot = fitres.resfunc3(xpts[:], fr, Qr, Qc_hat, a, phi, tau)
plt.plot(fit_plot.real, fit_plot.imag, 'r-',label='$f_r$={:.4f} MHz\n $\kappa$={:.4f} MHz\n$\kappa_c$={:.4f} MHz'.format(fr,fr/Qr, fr/(2*Qc)))
plt.axhline(0, color='grey')
plt.axvline(0, color='grey')
plt.legend()

# plt.savefig(os.path.join(summary_dir, f'resonator_spectroscopy_fine_lower_power_qi0.png'))

## Resonator Power Spectroscopy 

Find a good value for gain to park your readout at until you run readout optimization. From the 2D sweep that is produced, choose a value for gain that is right before the resonator punches out. Want to choose a high value for gain because we want to be in the shot noise limited regime which increases our signal:noise ratio. 

In [ ]:
# f_off: center scan at res_freq - f_off
q1 = 1

#params={'rng':100,'max_gain':1, 'span':10,"f_off":3,'expts_gain':25,'expts':100,'reps':0.5} #buffer
params={'start_gain':0.001,'step_gain':0.0004,'expts_gain':50,'span':10,"f_off":-1,'expts':2000,'reps':1000,'log':False} #waste
# params={'rng':100,'max_gain':0.5, 'span':50,'expts_gain':10,'f_off':10,'expts':200,'reps':1} #waste:
rpowspec=meas.ResSpecPower(cfg_dict, qi=qi, params=params, live_plot=False)

In [ ]:
# only update chi if the fit in the previous cell looks good. 
update=False
if update:
    auto_cfg = config.update_readout(cfg_path, 'lamb', rpowspec.data['lamb_shift'], qi)

In [ ]:
plt.pcolormesh(rpowspec.data['xpts'], rpowspec.data['gain_pts'], rpowspec.data['phases_corrected'], cmap="viridis", shading="auto", rasterized=True)

phases_corr = np.zeros_like(rpowspec.data['phases'])
f0s = []
kappas = []
kappa_cs = []
for i in range(rpowspec.data['gain_pts'].shape[0]):
    unwrapped = np.unwrap(rpowspec.data['phases'][i,:])
    polyfit = np.polyfit(rpowspec.data['xpts'][:100], unwrapped[:100], 1)
    unwrapped -= np.polyval(polyfit, rpowspec.data['xpts'])
    phases_corr[i,:] = unwrapped

    IQ = rpowspec.data['amps'][i,:]*np.exp(1j*unwrapped)
    try:
        popt = fitres.finefit(rpowspec.data['xpts'], IQ, [7742],p0=[0,0,0])
        if popt[0] > 7740:
            f0s.append(popt[0])
            kappas.append(popt[0]/popt[1])
            kappa_cs.append(popt[0]/popt[6]/2)
        else:
            print('fit failed, resonance frequency too far from expected value')
            f0s.append(np.nan)
            kappas.append(np.nan)
            kappa_cs.append(np.nan)
    except:
        print('error fitting')
        f0s.append(np.nan)
        kappas.append(np.nan)
        kappa_cs.append(np.nan)
        pass
    

    if rpowspec.data['gain_pts'][i] == 0.0086:
        plt.figure()
        plt.plot(IQ.real, IQ.imag, 'o-')
        plt.plot(fitres.resfunc3(rpowspec.data['xpts'], *popt[:6]).real, fitres.resfunc3(rpowspec.data['xpts'], *popt[:6]).imag, 'r-')
        fr_idx = np.argmin(abs(rpowspec.data['xpts']-popt[0]))
        
        plt.plot(fitres.resfunc3(popt[0], *popt[:6]).real, fitres.resfunc3(popt[0], *popt[:6]).imag, 'rx', label='fit at fr')
        plt.legend()

        plt.axis('equal')

plt.figure()
plt.pcolormesh(rpowspec.data['xpts'], rpowspec.data['gain_pts'], phases_corr, cmap="viridis", shading="auto", rasterized=True)
fig,ax = plt.subplots(nrows=1,ncols=3, figsize=(14,5))
ax[0].plot(rpowspec.data['gain_pts'], f0s, 'o-')
ax[1].plot(rpowspec.data['gain_pts'], kappas, 'o-')
ax[2].plot(rpowspec.data['gain_pts'], kappa_cs, 'o-')


print(list(zip(rpowspec.data['gain_pts'], f0s)))

# plt.xlim([0.01,0.0125])

### Save gain values 

In [ ]:
# assign the resonator gain to the results config file
gain_values = [0.1,0.01]
for i, qi in enumerate(qubit_list):
    auto_cfg = config.update_readout(cfg_file, 'gain', gain_values[i], qi)

# Qubit Spectroscopy

## General search, specify width 

style options: fine, medium, coarse, huge (will change scan width and power)

Uses config values of low_gain (gain to use for finest scan), which sets overall gain for device and spec_gain (set indepedently for each qubit) to decide how much power to apply

You may also just want to do this fully manually by specifiying params. 

In [ ]:
update=True

qubit_list = [1]

for qi in qubit_list: 
    # Default params, just specify style 
    #qspec=meas.QubitSpec(cfg_dict, qi=qi, style='huge')

    # Different examples of params you might give; frequency can be specified as start and span or if no start given, center is f_ge from config
    #qspec=meas.QubitSpec(cfg_dict, qi=qi, style='coarse', params={'span':500,'start':3000, 'expts':1000, 'gain':0.2})
    #params={'start':3825, 'span':1000, 'gain':1, 'expts':1000}
    #params={'span':50,'expts':200,'gain':0.06,'sep_readout':True, 'length':3, 'readout_length':10, 'reps':10000}
    params={'span':30,'expts':200,'gain':0.01,'reps':1000,'length':30, 'readout_length':5,'sep_readout':True}

    qspec=meas.QubitSpec(cfg_dict, qi=qi, style='fine', params=params)
    if update and qspec.status: 
        auto_cfg = config.update_qubit(cfg_path, 'f_ge', qspec.data["best_fit"][2], qi)
        auto_cfg = config.update_qubit(cfg_path, 'kappa',qspec.data["best_fit"][3], qi)
    elif update:
        print(f'Bad qubit! qi={qi}')

In [ ]:
auto_cfg = config.update_qubit(cfg_path, 'f_ge', qspec.data["best_fit"][2], qi)
auto_cfg = config.update_qubit(cfg_path, 'kappa',qspec.data["best_fit"][3], qi)

## Stark (still getting it working)

In [ ]:
qi=1
params={'df_stark':0, 'max_stark_gain':0.1,'min_stark_gain':0.003, 'df':-36,'span':85, 'stark_expts':50,"length":10,"stark_length":25, 'reps':500,"final_delay":100,'stark_chan':0}
stark_spec=meas.StarkSpec(cfg_dict, qi=qi, style='medium', params=params)

n_photon_conversion = stark_spec.data['n']

this returns term $k g^2$, which we can then use to calculate the number of photons in the resonator during readout, as well as frequency offset during readout. 

In [ ]:
auto_cfg_model = config.load(cfg_file_path[:-4] + '_model.yml')
auto_cfg= config.load(cfg_file_path)

k = stark_spec.data['ng2']/auto_cfg_model['g_chi'][qi]
nphotons = auto_cfg['device']['readout']['gain'][qi]**2*k

## Power sweep

In [ ]:
qubit_list=[1]
#qubit_list=np.arange(6)

for qi in qubit_list:
    # params={'start':3700,'span':200,'expts':300}
    #qspec_pow = meas.QubitSpecPower(cfg_dict, qi=qi, style='', params=params)
    qspec_pow = meas.QubitSpecPower(cfg_dict, qi=1, style='', 
                                    params={'start':4369.4,
                                            'span':3,
                                            'rng': 200,
                                            'reps':50,
                                            'rounds':5,
                                            'expts':200,
                                            'expts_gain':20, 
                                            'max_gain':0.1, 
                                            'length':1,'sep_readout':True})

In [ ]:
print(qspec_pow.data.keys())
phase_array = qspec_pow.data['phases']
amp_array = qspec_pow.data['amps']
gains = qspec_pow.data['gain_pts']


for i in range(phase_array.shape[0]):
    unwrapped_phase = np.unwrap(phase_array[i,:])
    linear_fit = np.polyfit(qspec_pow.data['xpts'][:10], unwrapped_phase[:10],1)
    unwrapped_phase_subtracted = np.unwrap(phase_array[i,:]) - np.polyval(linear_fit, qspec_pow.data['xpts'])
    phase_array[i,:] = -1*unwrapped_phase_subtracted

plt.figure(figsize=(9,6))
plt.pcolormesh(qspec_pow.data['xpts'], gains, phase_array, shading='auto')
plt.ylabel('DAC gain')
plt.xlabel('pump frequency (MHz)')
plt.colorbar(label='Phase shift of buffer resonator (rad)')
plt.gca().set_yscale('log')

plt.figure()
plt.pcolormesh(qspec_pow.data['xpts'], gains, amp_array, shading='auto')
plt.ylabel('DAC gain')
plt.xlabel('pump frequency (MHz)')

plt.colorbar(label='Magnitude (ADC units)')
plt.gca().set_yscale('log')

plt.figure()
plt.pcolormesh(qspec_pow.data['xpts'], gains, qspec_pow.data['avgi'], shading='auto')
plt.gca().set_yscale('log')
plt.ylabel('DAC gain')
plt.xlabel('pump frequency (MHz)')
plt.colorbar(label='Average I (ADC units)')


### Sweep pulse length 
Useful when t1 low

sep_readout = False measures at same time as probe pulse (default is True)

In [ ]:
length = [1,3,5,10]
for l in length:
    for qi in qubit_list:
        params={'start':3000,'span':600,'reps':1500,'expts':1200, 'max_gain':0.4, 'length':1,'sep_readout':True}
        params={'start':4000,'span':200,'expts':300,'length':l}
        qspec_pow = meas.QubitSpecPower(cfg_dict, qi=qi, style='', params=params)
        #qspec_pow = meas.QubitSpecPower(cfg_dict, qi=qi, style='', params=params)

### Narrow scan

In [ ]:
qq=[]

qubit_list = np.arange(6)
#qubit_list=[2]
for qi in qubit_list:
    qspec_pow = meas.QubitSpecPower(cfg_dict, qi=qi, style='narrow')#, live_plot=True)
    
    # Nice for pretty pics once you have T1 measurement. 
    #qspec_pow = meas.QubitSpecPower(cfg_dict, qi=qi, style='fine', params={'length':'t1','max_gain':1})
    qq.append(qspec_pow)

# When measuring many qubits, can do a bunch of color plots this way
#handy.plot_many(qq, title='Qubit Power Amps 0.6-0.2', save_path=cfg_dict['expt_path'], yax='log', chan='amps')
#handy.plot_many(qq, title='Qubit Power Phase 0.6-0.2', save_path=cfg_dict['expt_path'], yax='log', chan='phases')

### Multiple wide scans looking for qubit

In [ ]:
qubit_list = np.arange(3)
qubit_list=[1]

span = 250
start_all = [2000, 3100, 3400]
end_all = [4780, 3700, 4250]
sensitivities = [0.8, 0.4, 0.2]

d = []
for qi in tqdm(qubit_list, desc='Qubit Number'):
    starts = np.arange(start_all[qi], end_all[qi], span)
    qresults = []
    for start in tqdm(starts, desc=f'Start Frequency Sweep'):
        q_res = meas.QubitSpecPower(
            cfg_dict, 
            qi=qi, 
            style='coarse', 
            params={
                'max_gain':0.8,
                'start':start,
                'span':span,
                'rng':100,
                'reps':800}
        )#,'start':3000,'span':300'})
        qresults.append(q_res)
    
    # Handy plot
    handy.plot_many_limited(
        qresults, 
        title=f'Qubit Power for qubit {qi}', 
        save_path=cfg_dict['expt_path'],
        yax='log', 
        chan='amps', 
        individial_fig_size= (6,6), 
        xlabel='Frequency (MHz)',
        sensitivity =  sensitivities[qi],
        save = False,
    )
    d.append(q_res)
    plt.show()

# Coherent scans

## Fast tuneup

Options are: 

first_time: assume we don't know t1 time,don't have single shot working

single: do single shot readout optimization 

readout: set readout frequency based on resonator fit

In [ ]:
qubit_list = np.arange(6)
#qubit_list=[1]


#qubit_list=np.delete(qubit_list, [5,13])
# Worst issue with this right now is when the qubit frequency is not correct and readout is bad;
# gets stuck doing ramsey/spectroscopy forever. In this case, cancel it and go back to find qubits, 
# try changing readout gain. 
plt.rcParams.update({'font.size': 11})
for qi in qubit_list: 
    qubit_tuning.tune_up_qubit(qi, cfg_dict, first_time=False, single=False, readout=True)

## Time tracking
fast = True only measures T2 and T1, otherwise does full set of scans.

In [ ]:
scan_length = 6 #  hours 

qubit_list = [1]
tt, csv_pth, tt_stats = time_tracking(qubit_list, cfg_dict, display=False, total_time=scan_length, fast=True, bf_client=bf)

In [ ]:
qi=1
scan_length = 1 #  hours 
qubit_list = [qi]

for i in range(10):
    measure_func.measure_temp(cfg_dict, qi, bf_client=bf, temp=40)
    tt, csv_pth, tt_stats, tracker_id = time_tracking(qubit_list, cfg_dict, display=False, total_time=scan_length, fast=True, bf_client=bf)

In [ ]:
qi=1
pop, temp, rabi_no, rabi = measure_func.measure_temp(cfg_dict, qi, bf_client=bf, temp=100)

In [ ]:
from slab_qick_calib.analysis import collections
tt, tt_ave = collections.process_data(tt)
tt=collections.add_losses(tt)
from slab_qick_calib.analysis import allan 
#allan.perform_analysis(tt,param='t1', qubit_list=[1,2,3,4,5])
allan.perform_analysis(tt,param='Gamma1')

In [ ]:
collections.plot_all(tt, param_keys=['t1', 'tphi', 'f_ge', 't2_r2'])

In [ ]:
collections.plot_all(tt, param_keys=['Gamma1', 'Gamma2', 'Gammaphi', 't2_r2'], plot_time=True)

In [ ]:
collections.plot_violin(tt, param_keys=['Gamma1', 'Gammaphi', 'q', 'f_ge'])
#collections.plot_violin(tt, param_keys=['Gamma1', 'Gammaphi', 'q', 'f_ge'], qubit_list=[1,2,4], csv_path=csv_pth)

In [ ]:
collections.plot_violin(tt, param_keys=['Gamma1', 'Gammaphi', 'q', 'f_ge'], qubit_list=[1,2,3,4,5], csv_path=csv_pth)

In [ ]:
collections.plot_violin(tt, param_keys=['Gamma1'], qubit_list=[2])

In [ ]:
qubit_params.stats(cfg_path, tt_stats, qubit_list)

## Rabi

### Amplitude

Uses gain/sigma set in pulses part of config

In [ ]:
qubit_list = [1]
update=True

for qi in qubit_list: 
    #amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi)#, disp_kwargs={'show_hist':True})
    
    # Fully customized version
    # amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi, 
    #                                params={'reps':2000,
    #                                        'pulse_type':'gauss',
    #                                        'expts':100})
    params = {'reps':2000}
    amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi, params=params)
    if update and amp_rabi.status:
        config.update_qubit(cfg_path, ('pulses','pi_ge','gain'), amp_rabi.data['pi_length'], qi) 

#### Amp Chevron

In [ ]:
d2=[]
qubit_list = [1]
for qi in qubit_list: 
    amp_rabi_chevron = meas.RabiChevronExperiment(cfg_dict,qi=qi, params={'span_f':2, "max_gain":1})#, params={'span_f':150,'sigma':0.25,'expts_f':100,'expts':100})

    #amp_rabi_chevron = meas.RabiChevronExperiment(cfg_dict,qi=qi, params={'span_f':10, 'expts_f':20, 'checkEF':True})#, live_plot=True)
    d2.append(amp_rabi_chevron)

#handy.plot_many(d2, title='Rabi Chevron Phase', save_path=cfg_dict['expt_path'], chan='phases')

### Length -- Uses const pulses so do not use to set up pi pulses

Cannot do fast sweep with gaussian pulses due to multiplying qickparams issues; so need the "loop:True" for those, will make everything slower. 

In [ ]:
qubit_list = [1]
#qubit_list = np.arange(6)
for qi in qubit_list: 
    # Needs to have params of sweep: length and type: cons
    len_rabi = meas.RabiExperiment(cfg_dict,qi=qi, params={'sweep':'length', 'pulse_type':'const','reps':3000,'expts':100})
    #len_rabi = meas.RabiExperiment(cfg_dict,qi=qi, params={'sweep':'length', 'pulse_type':'gauss','sigma':0.05, 'loop':True, 'gain':0.05})
    #len_rabi = meas.RabiExperiment(cfg_dict,qi=qi, params={'sweep':'length', 'pulse_type':'const', 'max_length':1.2, 'gain':0.5,'expts':300})

#### Length Chevron

In [ ]:
#qubit_list = np.arange(6)
qubit_list=[1]
for qi in qubit_list: 
    #len_rabi_chevron = meas.RabiChevronExperiment(cfg_dict,qi=qi, params={'sweep':'length',"pulse_type":"const", 'max_length':1.2})
    #len_rabi = meas.RabiChevronExperiment(cfg_dict,qi=qi, params={'sweep':'length',"pulse_type":"const", 'length':0.2205, 'expts_f':40})
    len_rabi = meas.RabiChevronExperiment(cfg_dict,qi=qi, params={'sweep':'length',"pulse_type":"gauss", 'sigma':0.05, 'gain':0.25, 'span_f':50,"loop":True})

## Ramsey

In [ ]:
qubit_list = np.arange(6)
qubit_list=[1]

update = True

for qi in qubit_list:
    # t2r = meas.T2Experiment(cfg_dict, qi=qi, max_err=10)

    # Manually configured
    t2r = meas.T2Experiment(cfg_dict, qi=qi, max_err=10)#, disp_kwargs={'show_hist':True})
    #t2r = meas.T2Experiment(cfg_dict, qi=qi, max_err=10, params = {'ramsey_freq':1.1,'expts':100, 'span':10,'start':0.01})
    if t2r.status and update:
        config.update_qubit(cfg_path, 'f_ge', t2r.data['new_freq'], qi)
        auto_cfg = config.update_qubit(cfg_path, 'T2r', t2r.data['best_fit'][3], qi, rng_vals=[1.5, max_t1], sig=2)
    else:
        print('T2 Ramsey fit failed')

### Use Ramsey to recenter

In [ ]:
qubit_list = [0]
for qi in qubit_list:
    status = qubit_tuning.recenter(qi,cfg_dict)            

### Ramsey coherence

In [ ]:
t1= qubit_tuning.get_coherence(meas.RamseyExperiment, qi, cfg_dict,par='T2r')

## T1

If it's the first time, also set T2r and T2e as guesses 


In [ ]:
update=True
first_time=False

qubit_list = np.arange(1)
qubit_list=[1]
for qi in qubit_list:
    t1 = meas.T1Experiment(cfg_dict, qi=qi, params={'reps':1000})
    #t1 = meas.T1Experiment(cfg_dict, qi=qi, params={'reps':1000,'span':0, 'start':20})
    #t1 = meas.T1Experiment(cfg_dict, qi=qi, disp_kwargs={'show_hist':True})

    if update: t1.update(first_time=first_time)

### T1 coherence

Runs scan until scan is properly configured to be sensitive to T1

In [ ]:
qi=0
qubit_tuning.get_coherence(meas.T1Experiment,qi=qi,cfg_dict=cfg_dict,par='T1')

### Continuous scan at single point
Times do not seem to be accurate right now 

In [ ]:
qi=0
t1cont = meas.T1ContExperiment(cfg_dict,qi=qi, params={'shots':120000})

In [ ]:
t1cont.display(filter_type='boxcar')

In [ ]:
len(t1cont.data['t1_estimates'])
t1_data = t1cont.data['t1_estimates']

from slab_qick_calib.analysis import time_series
sampling_rate = 1/ float(t1cont.data['times'][2]-t1cont.data['times'][1])
print(f'Sampling rate is {sampling_rate} Hz')
nperseg = min(2048, int(2 ** np.floor(np.log2(len(t1_data) / 4))))

time_series.analyze_qubit_psd(
    t1_data, fs=sampling_rate, nperseg=nperseg
)

plt.figure()
plt.plot(t1cont.data['times'], t1cont.data['t1_estimates'], '.-')

In [ ]:
from slab_qick_calib.analysis import allan 

allan.perform_analysis(t1cont.data['times'], t1cont.data['t1_estimates'],'hi')

In [ ]:

nan_inds = np.where(np.isnan(t1cont.data['t1_estimates']))[0]

times = np.delete(t1cont.data['times'], nan_inds)
t1_data = np.delete(t1cont.data['t1_estimates'], nan_inds)

In [ ]:
allan.perform_analysis(times, t1_data,'hi')

In [ ]:
float(t1cont.data['times'][2]-t1cont.data['times'][1])

## Echo and more

In [ ]:
qubit_list = np.arange(6)
qubit_list=[1]
update=True
for qi in qubit_list:
    # Need to have experiment type set to echo
    t2e = meas.T2Experiment(cfg_dict, qi=qi, params={'experiment_type':'echo'})
    if t2e.status and update:
        auto_cfg = config.update_qubit(cfg_path, 'T2e', t2e.data['best_fit'][3], qi,sig=2, rng_vals=[1.5, max_t1*2])

### More pi / CPMG
Breaks above 12 right now; need to make it a python loop 

In [ ]:
qubit_list = np.arange(3)
qubit_list=[3]
update=True
nums_pi = [1,2,3,5,8,12]
t2_list = []
for qi in qubit_list:
    for pi in nums_pi:
        # Need to have experiment type set to echo
        #params=params={'experiment_type':'cpmg','num_pi':pi, 'span':400, 'ramsey_freq':0.02, 'reps':500}
        params=params={'experiment_type':'cpmg','num_pi':pi,}
        if pi>1:
            params['span']= t2e.data['fit_avgi'][3]*3.5
            params['ramsey_freq']=1.5/t2e.data['fit_avgi'][3]


        t2e = meas.T2Experiment(cfg_dict, qi=qi, params=params)
        t2_list.append(t2e.data['best_fit'][3])

plt.figure()
plt.plot(1/np.array(nums_pi), t2_list, 'o-')
plt.xlabel('1/Number of pi pulses')
plt.title(f'Q{qi}')
plt.ylabel('T2 (Âµs)')
plt.show()

### Get echo coherence

In [ ]:
qi=6
t2e = qubit_tuning.get_coherence(meas.RamseyEchoExperiment, qi, cfg_dict,'T2e')

## Feedback checks

In [ ]:
qi=0
# This makes sure that waits are set correctly for active reset so that you get the same value from the buffer as ... 
reset = meas.MemoryExperiment(cfg_dict, qi=qi, params={'shots':1, 'expts':10})

In [ ]:
# Don't do the active reset, just do the repeated measurement 
qi=1
shot = meas.RepMeasExperiment(cfg_dict, qi=qi, params={'shots':10000,'active_reset':False, 'setup_reset':False})
shot.check_reset()

In [ ]:

np.mean(shot.data['Ig'])
np.mean(shot.data['Ie'])

# Single Shot

In [ ]:
# Single shot 
qubit_list = np.arange(6)
qubit_list =[1]

for qi in qubit_list:
    shot=meas.HistogramExperiment(cfg_dict, qi=qi,params={'shots':20000})

    # Configure number of shots
    #shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':30000})
    shot.update()

In [ ]:
# compute the percentage of the shots in the excited state that are above the threshold and the percentage of shots in the ground state that are below the threshold. This gives you an idea of how well separated your histogram is and how good your readout fidelity is.
threshold = shot.data['thresholds'][0]
Ig = shot.data['Ig']
Ie = shot.data['Ie']
fidelity_g = np.mean(Ig < threshold)
fidelity_e = np.mean(Ie > threshold)
print(f'Readout fidelity for ground state: {fidelity_g:.4f}')
print(f'Readout fidelity for excited state: {fidelity_e:.4f}')


### Adjust reps for fidelity 

In [ ]:
max_inc = 15 # dont' let it do more than 15x standard number of reps so that things don't take forever
qubit_list = [1]

auto_cfg = config.load(cfg_path)
for qi in qubit_list:
    config.update_readout(cfg_path, 'reps', 1/auto_cfg['device']['readout']['fidelity'][qi]**1.5, qi, rng_vals=[1,max_inc]);

## Readout opt

### General sweep

Runs single shot experiments for many readout lengths, frequencies, gains and compares fidelity

low_gain=True chooses lowest gain/readout length within a few percent of maximum gain (often readout fidelity fairly flat as a function of gain at higher gain values) 

style='fine' varies parameters by 20%, style='' varies by 2x

In [ ]:
update=True
low_gain=False

qubit_list=np.arange(1,6)
qubit_list=[1]

#params = {'expts_f':1, 'expts_gain':10, 'expts_len':10,'shots':10000}
#params = {'expts_f':1, 'expts_gain':10, 'expts_len':1}
params = {'expts_f':10, 'expts_gain':1, 'expts_len':10,'shots':10000}
#params = {'expts_f':1, 'expts_gain':5, 'expts_len':5,'shots':10000}

# Specify exact ranges to use  
#params = {'expts_f':1, 'expts_gain':9, 'expts_len':9,'start_gain':0.45, 'span_gain':0.05,'start_len':2, 'span_len':5}

for qi in qubit_list: 
    shotopt=meas.SingleShotOptExperiment(cfg_dict, qi=qi,params=params, display=True, style="fine")
    shotopt.analyze(low_gain=low_gain)
    if update: shotopt.update(cfg_dict['cfg_file'])

    shot=meas.HistogramExperiment(cfg_dict, qi=qi)
    shot.update()

### Run optimization until it converges

In [ ]:
qubit_list=np.arange(3)
qubit_list=[1]
params = {'expts_f':1, 'expts_gain':5, 'expts_len':5}

# do_res also runs res spec and resets readout frequency that way each round. 
qubit_tuning.meas_opt(cfg_dict, qubit_list, params, do_res=True)

### Vary trig_offset to see if it changes fidelity. 

In [ ]:
# Single shot 
qubit_list = np.arange(3)
qubit_list =[0]
trig_list = np.linspace(0.2,0.6,12)
fids =[]
for qi in qubit_list:
    for trig in trig_list: 
        config.update_readout(cfg_path, 'trig_offset', trig, qi);
        shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':20000, 'trigger':trig}, progress=False, display=False)
        fids.append(float(shot.data['fids'][0]))

### Play with LO freq (when using qick for LO)

In [ ]:
auto_cfg = config.load(cfg_path)
start_freq = auto_cfg.device.readout.frequency[qi]
start_mixer = auto_cfg.hw.soc.lo.mixer_freq[qi]
rng = np.linspace(-1000,1000,11)
fids=[]
for qi in qubit_list: 
    for r in rng: 
        config.update_lo(cfg_path, 'mixer_freq', start_mixer+r, qi)
        config.update_readout(cfg_path, 'frequency', start_freq-r, qi)
        shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':20000})
        fids.append(shot.data['fids'][0])

config.update_lo(cfg_path, 'mixer_freq', start_mixer, qi)
config.update_readout(cfg_path, 'frequency', start_freq, qi)

#### Play with LO power

In [ ]:
qi = 0
gain_vals = [0.0375,0.05] 
fids = []
for gain in gain_vals:
    config.update_lo(cfg_path, 'gain', gain, qi)
    shotopt=meas.SingleShotOptExperiment(cfg_dict, qi=qi,params={'npts_f':5, 'npts_gain':5, 'npts_len':5})
    fids.append(shot.data['fids'][0])

# 4WM modified from single-shot

In [ ]:
pump_guess = 4373.5 + 7741.31 - 6917.1311
pump_guess

## single-set params

In [ ]:
smpd4wm.data['p_e']

In [ ]:
# 4WM Histogram 
smpd4wm=SMPD4WMHistogramExperiment(cfg_dict, qi=1,params={'do_buffer':True, 'gain_b':0.01, 'gain_p':0.4, 'f_p':5179.74, 'f_b':6917.3, 't_p':10, 't_b':10, 'shots':10000,'active_reset':True})


smpd4wm_dark=SMPD4WMHistogramExperiment(cfg_dict, qi=1,params={'do_buffer':False, 'gain_b':0.01, 'gain_p':0.4, 'f_p':5179.74, 'f_b':6917.3, 't_p':10, 't_b':10, 'shots':10000,'active_reset':False})

print(smpd4wm.data['p_e'])
print(smpd4wm_dark.data['p_e'])

In [ ]:
"""
Single-Shot Readout Experiment Module
=====================================

This module implements single-shot readout experiments for quantum processors.
It allows for discrimination between quantum states (g, e, and optionally f)
by collecting statistics on readout signals and analyzing their distributions.

The module contains two main classes:
- HistogramProgram: Defines the quantum pulse sequence for the experiment
- HistogramExperiment: Manages experiment execution, data acquisition, and analysis

Key features:
- Configurable readout parameters (frequency, gain, length)
- Support for ground, excited, and second excited state measurements
- Automatic data analysis with Gaussian fitting
- Visualization of readout histograms and IQ distributions
- Calculation of readout fidelity and optimal discrimination thresholds
- Support for active qubit reset and verification
"""

from slab_qick_calib.experiments.general.qick_program import QickProgram, QickProgram2Q
from slab_qick_calib.experiments.general.qick_experiment import QickExperiment
from slab_qick_calib.experiments.general.qick_experiment_2q import QickExperiment2Q
from slab_qick_calib.calib import readout_helpers as helpers
from slab_qick_calib.exp_handling.datamanagement import AttrDict

class SMPD4WMHistogramProgram(QickProgram):
    """
    Quantum pulse sequence program for single-shot readout experiments.

    This class defines the pulse sequence for measuring the qubit state
    in a single shot. It can be configured to prepare the qubit in the
    ground (g), excited (e), or second excited (f) state before readout.

    Parameters
    ----------
    soccfg : dict
        SOC configuration dictionary
    final_delay : float
        Final delay time after readout
    cfg : dict
        Configuration dictionary containing experiment parameters.
        This dictionary is expected to have the following keys:
        - `expt.shots`: Number of shots in the experiment
        - `expt.frequency`: Readout frequency
        - `expt.gain`: Readout gain
        - `expt.readout_length`: Length of the readout pulse
        - `expt.qubit`: List of qubit indices (e.g., `[0]`)
        - `device.readout.phase`: Readout phase from device config
        - `device.qubit.f_ge`: Qubit g-e transition frequency
        - `expt.pulse_f` (optional): Boolean to enable pulsing to the f-state
        - `device.qubit.f_ef` (optional): Qubit e-f transition frequency
        - `expt.active_reset` (optional): Boolean to enable active reset
    """

    def __init__(self, soccfg, final_delay, cfg, final_wait=0):
        super().__init__(soccfg, final_delay=final_delay, cfg=cfg)#, final_wait=final_wait)

    def _initialize(self, cfg):
        """
        Initialize the program with the given configuration.

        Sets up the experiment loop, readout parameters, and pulse definitions.

        Parameters
        ----------
        cfg : dict
            Configuration dictionary
        """
        cfg = AttrDict(self.cfg)
        # Set up experiment loop for the specified number of shots
        self.add_loop("shotloop", cfg.expt.shots)

        # Configure readout parameters
        q = cfg.expt.qubit[0]
        self.frequency = cfg.expt.frequency
        self.gain = cfg.expt.gain
        self.phase = cfg.device.readout.phase[q]
        # Set phase based on whether active reset is enabled
        # if cfg.expt.active_reset or cfg.expt.remeas:
        #     self.phase = cfg.device.readout.phase[cfg.expt.qubit[0]]
        # else:
        #     self.phase = 0
        self.readout_length = cfg.expt.readout_length

        # Initialize the base program with readout configuration
        super()._initialize(cfg, readout="standard")
        self.declare_gen(ch = cfg.hw.soc.dacs.readout.ch[0])
        
        # self.add_gauss(
        #     ch = cfg.hw.soc.dacs.readout.ch[0], 
        #     name = 'gauss_buffer',
        #     sigma = cfg.expt.sigma,
        #     length = 6*cfg.expt.sigma,
        #     maxv = cfg.expt.gain_b,
        #     even_length = True)
        
        # self.add_pulse(
        #     ch = cfg.hw.soc.dacs.readout.ch[0],
        #     name = 'pulse_buffer',
        #     #ro_ch = cfg.hw.soc.adcs.readout.ch[1],
        #     style = 'flat_top',
        #     freq = cfg.expt.f_b,
        #     phase = cfg.device.readout.phase[0],
        #     gain = cfg.expt.gain_b,
        #     length = cfg.expt.t_b,
        #     envelope = 'gauss_buffer',
        # )
        
        # self.add_gauss(
        #     ch = cfg.hw.soc.dacs.qubit.ch[0], 
        #     name = 'gauss_pump',
        #     sigma = cfg.expt.sigma,
        #     length = 6*cfg.expt.sigma,
        #     maxv = cfg.expt.gain_p,
        #     even_length = True)
        
        # self.add_pulse(
        #     ch = cfg.hw.soc.dacs.qubit.ch[0],
        #     name = 'pulse_pump',
        #     #ro_ch = cfg.hw.soc.adcs.readout.ch[1],
        #     style = 'flat_top',
        #     freq = cfg.expt.f_p,
        #     phase = 0,
        #     gain = cfg.expt.gain_p,
        #     length = cfg.expt.t_p,
        #     envelope = 'gauss_pump',
        # )


        qubit_pulse = {
            "freq": cfg.expt.f_p,
            "gain": cfg.expt.gain_p,
            "type": 'const',
            "sigma": cfg.expt.t_p,
            "phase": 0,
        }
        super().make_pulse(qubit_pulse, "pulse_pump")

        buffer_pulse = {
            "freq": cfg.expt.f_b,
            "gain": cfg.expt.gain_b,
            "type": 'const',
            "sigma": cfg.expt.t_b,
            "phase": 0,
            "chan": cfg.hw.soc.dacs.readout.ch[0]
        }
        super().make_pulse(buffer_pulse, "pulse_buffer")

        # Define pi pulses for state preparation
        super().make_cfg_pulse(cfg.expt.qubit[0], cfg.device.qubit.f_ge, "pi_ge")
        if cfg.expt.pulse_f:
            super().make_cfg_pulse(cfg.expt.qubit[0], cfg.device.qubit.f_ef, "pi_ef")

        # Add initial delay for tProc setup
        self.delay(0.5)

    def _body(self, cfg):
        """
        Define the main body of the pulse sequence.

        This includes state preparation pulses, readout pulse, and triggers.

        Parameters
        ----------
        cfg : dict
            Configuration dictionary
        """
        cfg = AttrDict(self.cfg)
        # Configure readout
        if self.adc_type == "dyn":
            self.send_readoutconfig(ch=self.adc_ch, name="readout", t=0)

        # Perform active reset if enabled
        # if cfg.expt.active_reset:
        #     self.reset(cfg.expt.reset)
            #self.delay_auto(10)

        '''
        # Apply pi pulse to prepare excited state if requested
        if cfg.expt.pulse_e:
            self.pulse(ch=self.qubit_ch, name="pi_ge", t=0)

        # Apply second pi pulse to prepare f state if requested
        if cfg.expt.pulse_f:
            self.pulse(ch=self.qubit_ch, name="pi_ef", t=0)
        '''

        if cfg.expt.pulse_e:
            self.pulse(ch=cfg.hw.soc.dacs.qubit.ch[0], name="pulse_pump", t=0)
        if cfg.expt.do_buffer:
            self.pulse(ch=cfg.hw.soc.dacs.readout.ch[0], name="pulse_buffer", t=0)
        
        
        # Add small delay before readout
        self.delay_auto(t=0.01, tag="wait")

        # Apply readout pulse and trigger data acquisition
        self.pulse(ch=self.res_ch, name="readout_pulse", t=0)
        if self.lo_ch is not None:
            self.pulse(ch=self.lo_ch, name="mix_pulse", t=0.0)
        self.trigger(ros=[self.adc_ch], ddr4=True, pins=[0], t=self.trig_offset)

        if cfg.expt.active_reset:
            #self.reset2(cfg.expt.reset)
            super().reset(cfg.expt.reset)
        if cfg.expt.remeas:
            self.repeated_measurement(5)

    def reset(self, i):
        """
        Reset the qubit to ground state.

        Parameters
        ----------
        i : int
            Reset index
        """
        cfg = AttrDict(self.cfg)

        # Perform reset sequence i times
        for n in range(i):
            # Apply readout pulse and trigger data acquisition
            self.pulse(ch=self.res_ch, name="readout_pulse", t=0)
            #if self.lo_ch is not None:
            #    self.pulse(ch=self.lo_ch, name="mix_pulse", t=0.0)
            self.trigger(ros=[self.adc_ch], ddr4=True, pins=[0], t=self.trig_offset)
            # Wait for readout to complete
            #self.wait(7)
            #self.delay(14)
            self.wait_auto(cfg.expt.read_wait)
            # Add extra delay for stability
            self.delay_auto(cfg.expt.read_wait + cfg.expt.extra_delay+10)
            

            # Read qubit state and conditionally apply π pulse
            # If I < threshold (qubit in |1⟩), apply π pulse to return to |0⟩
            # If I >= threshold (qubit in |0⟩), skip the π pulse
            self.read_and_jump(
                ro_ch=self.adc_ch,
                component="I",  # Use I quadrature for state discrimination
                threshold=cfg.expt.threshold,  # Threshold for state discrimination
                test="<",  # Skip to end of no pulse if I < threshold
                label=f"NOPULSE{n}",  # Jump to this label if I >= threshold
            )

            # Apply π pulse to flip qubit from |1⟩ to |0⟩
            self.pulse(ch=self.qubit_ch, name="pi_ge", t=0)
            # Small delay for pulse completion
            self.delay_auto(0.01)

            # Label for conditional jump target
            self.label(f"NOPULSE{n}")

    def reset2(self, i):
        """
        Reset the qubit to ground state.

        Parameters
        ----------
        i : int
            Reset index
        """
        cfg = AttrDict(self.cfg)

        # Perform reset sequence i times
        for n in range(i):
            if i>0:
            # Apply readout pulse and trigger data acquisition
                self.pulse(ch=self.res_ch, name="readout_pulse", t=0)
                #if self.lo_ch is not None:
                #    self.pulse(ch=self.lo_ch, name="mix_pulse", t=0.0)
                self.trigger(ros=[self.adc_ch], ddr4=True, pins=[0], t=self.trig_offset)
                # Wait for readout to complete

                self.wait_auto(cfg.expt.read_wait)
                # Add extra delay for stability
            self.delay_auto(cfg.expt.read_wait + cfg.expt.extra_delay)
            

            # Read qubit state and conditionally apply π pulse
            # If I < threshold (qubit in |1⟩), apply π pulse to return to |0⟩
            # If I >= threshold (qubit in |0⟩), skip the π pulse
            self.read_and_jump(
                ro_ch=self.adc_ch,
                component="I",  # Use I quadrature for state discrimination
                threshold=cfg.expt.threshold,  # Threshold for state discrimination
                test="<",  # Skip to end of no pulse if I < threshold
                label=f"NOPULSE{n}",  # Jump to this label if I >= threshold
            )

            # Apply π pulse to flip qubit from |1⟩ to |0⟩
            self.pulse(ch=self.qubit_ch, name="pi_ge", t=0)
            # Small delay for pulse completion
            self.delay_auto(0.01)

            # Label for conditional jump target
            self.label(f"NOPULSE{n}")


    def repeated_measurement(self, i):
        """
        Perform repeated measurement.

        Parameters
        ----------
        i : int
            Number of repetitions
        """
        cfg = AttrDict(self.cfg)
        for n in range(i):
            self.wait_auto(0.1)
            self.delay_auto(0.3)
            
            self.trigger(ros=[self.adc_ch], pins=[0], t=self.trig_offset)
            self.pulse(ch=self.res_ch, name="readout_pulse", t=0)
            if self.lo_ch is not None:
                self.pulse(ch=self.lo_ch, name="mix_pulse", t=0.0)
            self.delay_auto(0.01)

    def collect_shots(self, offset=0):
        """
        Collect and process the raw I/Q data from the experiment.

        Parameters
        ----------
        offset : float, optional
            Offset to subtract from the raw data

        Returns
        -------
        tuple
            (i_shots, q_shots) arrays containing I and Q values for each shot
        """
        for i, (ch, rocfg) in enumerate(self.ro_chs.items()):
            # Get raw IQ data
            iq_raw = self.get_raw()
            # Extract I values and flatten
            i_shots = iq_raw[i][:, :, 0, 0]
            i_shots = i_shots.flatten()
            # Extract Q values and flatten
            q_shots = iq_raw[i][:, :, 0, 1]
            q_shots = q_shots.flatten()

        return i_shots, q_shots


class SMPD4WMHistogramExperiment(QickExperiment):
    """
    Single-shot readout experiment for quantum state discrimination.

    This class manages the execution of single-shot readout experiments,
    including data acquisition, analysis, and visualization. It can measure
    the ground (g), excited (e), and optionally second excited (f) states.

    Parameters
    ----------
    cfg_dict : dict
        Configuration dictionary
    prefix : str, optional
        Prefix for experiment name and saved files
    progress : bool, optional
        Whether to show progress during acquisition
    qi : int, optional
        Qubit index
    go : bool, optional
        Whether to run the experiment immediately
    check_f : bool, optional
        Whether to measure the second excited state
    params : dict, optional
        A dictionary of parameters to override the default values.
        If not provided, the following defaults are used:
        - `shots`: 10000
        - `reps`: 1
        - `rounds`: 1
        - `readout_length`: from device config for the specified qubit
        - `frequency`: from device config for the specified qubit
        - `gain`: from device config for the specified qubit
        - `active_reset`: `False`
        - `check_e`: `True`
        - `check_f`: as passed to `__init__`
        - `qubit`: `[qi]`
        - `qubit_chan`: from hardware config for the specified qubit
        - `ddr4`: `False`
    display : bool, optional
        Whether to display the results after acquisition
    """

    def __init__(
        self,
        cfg_dict,
        prefix=None,
        progress=True,
        qi=0,
        go=True,
        check_f=False,
        params={},
        display=True,
        print=False,
    ):
        # Set default prefix if not provided
        if prefix is None:
            prefix = f"single_shot_qubit{qi}"

        # Initialize base experiment
        super().__init__(cfg_dict=cfg_dict, prefix=prefix, progress=progress, qi=qi)
    
        # Define default parameters
        params_def = dict(
            shots=10000,  # Number of shots per experiment
            reps=1,  # Number of repetitions
            rounds=1,  # Number of software averages
            readout_length=self.cfg.device.readout.readout_length[
                qi
            ],  # Readout pulse length
            frequency=self.cfg.device.readout.frequency[qi],  # Readout frequency
            gain=self.cfg.device.readout.gain[qi],  # Readout gain
            active_reset=False,  # Whether to use active reset
            reset=7,  # Reset index
            check_e=True,  # Whether to measure excited state
            check_f=check_f,  # Whether to measure second excited state
            qubit=[qi],  # Qubit index list
            qubit_chan=self.cfg.hw.soc.adcs.readout.ch[qi],  # Readout channel
            ddr4=False,  # Whether to use DDR4 memory
            remeas=False,  # Whether to do repeated measurement
            final_delay=self.cfg.device.readout.readout_length[qi],  # Final delay

            ##
            do_buffer = True,
            sigma = 0.02, # sigma of the gaussian envelope
            #"t_m": self.cfg.device.readout.readout_length[1], # length of the waste pulse [us]
            t_b = 2, # length of the buffer pulse [us]
            gain_b = self.cfg.device.readout.gain[0], # gain of the buffer pulse
            f_b = self.cfg.device.readout.frequency[0], # frequency of the buffer pulse
            t_p = 2, # length of the pump pulse [us]
            gain_p = 0.01, # gain of the pump pulse
            f_p = 4900, # frequency of the pump pulse 
        )

        # Merge default and user-provided parameters
        self.cfg.expt = {**params_def, **params}

        # Configure reset if active reset is enabled
        if self.cfg.expt.active_reset:
            super().configure_reset()

        if print:
            super().print()
            go = False
        # Run the experiment if requested
        if go:
            self.go(analyze=True, display=display, progress=progress, save=True)

    def acquire(self, progress=False, debug=False):
        """
        Acquire data for the single-shot experiment.

        This method collects single-shot data for the ground state and,
        if configured, the excited and second excited states.

        Parameters
        ----------
        progress : bool, optional
            Whether to show progress during acquisition
        debug : bool, optional
            Whether to print debug information

        Returns
        -------
        dict
            Dictionary containing the acquired data
        """
        data = dict()

        # Determine final delay based on configuration
        if "setup_reset" in self.cfg.expt and self.cfg.expt.setup_reset:
            final_delay = self.cfg.device.readout.final_delay[self.cfg.expt.qubit[0]]
        elif self.cfg.expt.active_reset:
            #final_delay = self.cfg.expt.final_delay
            final_delay=1
        else:
            final_delay = self.cfg.device.readout.final_delay[self.cfg.expt.qubit[0]]

        # Configure DDR4 parameters if enabled
        if self.cfg.expt.ddr4:
            # Each transfer (burst) is 256 decimated samples
            n_transfers = 1500000
            nt = n_transfers

        # Define scan configurations for different quantum states
        scan_configs = [
            {
                'name': 'ground',
                'pulse_e': False,
                'pulse_f': False,
                'data_keys': ('Ig', 'Qg', 'Igr', 'Qgr', 't_g', 'iq_ddr4_g'),
                'enabled': True,
                'acquire_kwargs': {}
            }
        ]
        
        # Add excited state scan if enabled
        if self.cfg.expt.check_e:
            scan_configs.append({
                'name': 'excited',
                'pulse_e': True,
                'pulse_f': False,
                'data_keys': ('Ie', 'Qe', 'Ier', 'Qer', 't_e', 'iq_ddr4_e'),
                'enabled': True,
                'acquire_kwargs': {}
            })

        # Add second excited state scan if enabled
        self.check_f = self.cfg.expt.check_f
        if self.check_f:
            scan_configs.append({
                'name': 'second_excited',
                'pulse_e': True,
                'pulse_f': True,
                'data_keys': ('If', 'Qf', 'Ifr', 'Qfr', 't_f', 'iq_ddr4_f'),
                'enabled': True,
                'acquire_kwargs': {}
            })

        # Loop through each scan configuration
        for scan_config in scan_configs:
                
            if debug:
                print(f"Acquiring {scan_config['name']} state data...")

            # Create configuration for this scan
            cfg = AttrDict(copy.deepcopy(dict(self.cfg)))
            cfg.expt.pulse_e = scan_config['pulse_e']
            cfg.expt.pulse_f = scan_config['pulse_f']

            # Create and configure histogram program
            kwargs = {"soccfg": self.soccfg, "final_delay": final_delay, "cfg": cfg}
            # if self.cfg.expt.active_reset: 
            #     kwargs["final_wait"] = None
            histpro = SMPD4WMHistogramProgram(**kwargs)

            # Configure DDR4 if enabled
            if self.cfg.expt.ddr4:
                self.im[self.cfg.aliases.soc].arm_ddr4(
                    ch=self.cfg.expt.qubit_chan, nt=n_transfers
                )

            # Acquire data for this scan
            acquire_kwargs = {
                'threshold': None,
                'progress': progress,
                **scan_config['acquire_kwargs']
            }
            
            iq_list = histpro.acquire(
                self.im[self.cfg.aliases.soc],
                **acquire_kwargs
            )

            # Extract data keys for this scan
            i_key, q_key, ir_key, qr_key, t_key, ddr4_key = scan_config['data_keys']

            # Use for active reset first 
            # # Store I/Q data
            # data[i_key] = iq_list[0][-1][:, 0]
            # data[q_key] = iq_list[0][-1][:, 1]

            # # Store reset data if active reset is enabled
            # if self.cfg.expt.active_reset or self.cfg.expt.remeas:
            #     data[ir_key] = iq_list[0][0:-1, :, 0]
            #     data[qr_key] = iq_list[0][0:-1, :, 1]

            # Use for active reset at the end
            # Store I/Q data
            data[i_key] = iq_list[0][0][:, 0]
            data[q_key] = iq_list[0][0][:, 1]

            # Store reset data if active reset is enabled
            if self.cfg.expt.active_reset or self.cfg.expt.remeas:
                data[ir_key] = iq_list[0][1:, :, 0]
                data[qr_key] = iq_list[0][1:, :, 1]

            # Get DDR4 data if enabled
            if self.cfg.expt.ddr4:
                iq_ddr4 = self.im[self.cfg.aliases.soc].get_ddr4(nt)
                t = histpro.get_time_axis_ddr4(self.cfg.expt.qubit_chan, iq_ddr4)
                data[t_key] = t
                data[ddr4_key] = iq_ddr4

        # Store data and return
        self.data = data
        return data

    def analyze(self, data=None, span=None, verbose=False, **kwargs):
        """
        Analyze the acquired single-shot data.

        This method processes the data to calculate readout fidelity,
        optimal thresholds, and fit parameters for state discrimination.

        Parameters
        ----------
        data : dict, optional
            Data dictionary to analyze (uses self.data if None)
        span : float, optional
            Span for histogram analysis
        verbose : bool, optional
            Whether to print detailed analysis information
        **kwargs : dict
            Additional keyword arguments

        Returns
        -------
        dict
            Dictionary containing the analyzed data and results
        """
        if data is None:
            data = self.data

        # Perform initial histogram analysis
        params, _ = helpers.analyze_single_shot_histograms(data=data, plot=False, span=span, verbose=verbose)
        data.update(params)

        # Perform detailed single-shot analysis with fitting
        try:
            # Fit single-shot data
            data2, p, paramsg, paramse2 = helpers.fit_single_shot(data, plot=False)

            # Update data with fit results
            data.update(p)
            data["vhg"] = data2["vhg"]
            data["histg"] = data2["histg"]
            data["vhe"] = data2["vhe"]
            data["histe"] = data2["histe"]
            data["paramsg"] = paramsg
            data["shots"] = self.cfg.expt.shots
            data['e_mean'] = p['e_mean']
            data['g_mean'] = p['g_mean']
            dv = self.data['ve'] - self.data['vg']
            data['e_norm'] = (self.data['e_mean']-self.data['vg'])/dv

            data['g_norm'] = (self.data['g_mean']-self.data['vg'])/dv
        except Exception as e:
            print(f"Fits failed: {str(e)}")
        
        qi = self.cfg.expt.qubit[0]
        threshold = self.cfg.device.readout.threshold[qi]
        i_shots = data['Ie']
        data['p_e'] = np.mean(i_shots > threshold, axis=0)

        return data

    def display(
        self,
        data=None,
        span=None,
        verbose=False,
        plot_e=True,
        plot_f=False,
        ax=None,
        plot=True,
        **kwargs,
    ):
        """
        Display the results of the single-shot experiment.

        This method creates visualizations of the single-shot data,
        including histograms and IQ distributions.

        Parameters
        ----------
        data : dict, optional
            Data dictionary to display (uses self.data if None)
        span : float, optional
            Span for histogram analysis
        verbose : bool, optional
            Whether to print detailed information
        plot_e : bool, optional
            Whether to plot excited state data
        plot_f : bool, optional
            Whether to plot second excited state data
        ax : list of matplotlib.axes, optional
            Axes for plotting
        plot : bool, optional
            Whether to create plots
        **kwargs : dict
            Additional keyword arguments

        Returns
        -------
        None
        """
        if data is None:
            data = self.data

        # Determine whether to save the figure
        if ax is not None:
            savefig = False
        else:
            savefig = True

        # Create histogram plots
        params, fig = helpers.analyze_single_shot_histograms(
            data=data,
            plot=plot,
            verbose=verbose,
            span=span,
            ax=ax,
            qubit=self.cfg.expt.qubit[0],
        )

        # Extract parameters
        fids = params["fids"]
        thresholds = params["thresholds"]
        angle = params["angle"]

        # Set experiment parameters if not already set
        if "expt" not in self.cfg:
            self.cfg.expt.check_e = plot_e
            self.cfg.expt.check_f = plot_f

        # Print detailed information if requested
        if verbose:
            print(f"ge Fidelity (%): {100*fids[0]:.3f}")

            if self.cfg.expt.check_f:
                print(f"gf Fidelity (%): {100*fids[1]:.3f}")
                print(f"ef Fidelity (%): {100*fids[2]:.3f}")
            print(f"Rotation angle (deg): {angle:.3f}")
            print(f"Threshold ge: {thresholds[0]:.3f}")
            if self.cfg.expt.check_f:
                print(f"Threshold gf: {thresholds[1]:.3f}")
                print(f"Threshold ef: {thresholds[2]:.3f}")

        # Show and save figure if requested
        if savefig:
            plt.show()
            self.save_fig(fig)

    def update(self, freq=True, fast=False, verbose=True):
        """
        Update configuration file with the results of the experiment.

        This method updates the readout parameters in the configuration file
        based on the analysis results.

        Parameters
        ----------
        cfg_file : str
            Path to the configuration file
        freq : bool, optional
            Whether to update frequency
        fast : bool, optional
            Whether to perform a fast update (skip some parameters)
        verbose : bool, optional
            Whether to print update information

        Returns
        -------
        None
        """
        qi = self.cfg.expt.qubit[0]
        cfg_file=self.config_file

        # Update readout parameters
        config.update_readout(
            cfg_file, "phase", self.data["angle"], qi, verbose=verbose
        )
        config.update_readout(
            cfg_file, "threshold", self.data["thresholds"][0], qi, verbose=verbose
        )
        config.update_readout(
            cfg_file, "fidelity", self.data["fids"][0], qi, verbose=verbose
        )

        # Update additional parameters if not in fast mode
        if not fast:
            config.update_readout(
                cfg_file, "sigma", self.data["sigma"], qi, verbose=verbose
            )
            config.update_readout(cfg_file, "tm", self.data["tm"], qi, verbose=verbose)

            # Update qubit tuned_up status based on fidelity
            if self.data["fids"][0] > 0.07:
                config.update_qubit(cfg_file, "tuned_up", True, qi, verbose=verbose)
            else:
                config.update_qubit(cfg_file, "tuned_up", False, qi, verbose=verbose)
                print("Readout not tuned up")

    def check_reset(self):
        """
        Check the performance of active reset.

        This method analyzes and visualizes the effectiveness of active reset
        by comparing the distributions before and after reset.

        Parameters
        ----------
        None

        Returns
        -------
        None
        """
        # Create histograms with specified number of bins
        n_bins = 75
        fig, ax = plt.subplots(2, 1, figsize=(6, 7))
        fig.suptitle(f"Q{self.cfg.expt.qubit[0]}")

        # Create ground state histogram
        vg, histg = helpers.create_histogram(self.data["Ig"], n_bins=n_bins)
        max_g = np.max(histg)
        ax[0].semilogy(vg, histg/max_g, color=BLUE, linewidth=2)
        ax[1].semilogy(vg, histg/max_g, color=BLUE, linewidth=2)

        # Create color palette for reset histograms
        b = sns.color_palette("ch:s=-.2,r=.6", n_colors=len(self.data["Igr"]))

        # Create excited state histogram
        ve, histe = helpers.create_histogram(self.data["Ie"], n_bins=n_bins)
        max_e = np.max(histe)
        ax[1].semilogy(ve, histe/max_e, color=RED, linewidth=2)
        fig2, ax2 = plt.subplots(2, len(self.data["Igr"])+1, figsize=(17, 7), sharex=True, sharey=True)
        # Plot reset histograms for ground state
        ax2[0,0].plot(self.data['Ig'], self.data['Qg'], '.', markersize=1, rasterized=True)
        ax2[1,0].plot(self.data['Ie'], self.data['Qe'], '.', markersize=1, rasterized=True)
        for i in range(len(self.data["Igr"])):
            ax2[0,i+1].plot(self.data['Igr'][i], self.data['Qgr'][i], '.', markersize=1, rasterized=True)
            ax2[1,i+1].plot(self.data['Ier'][i], self.data['Qer'][i], '.', markersize=1, rasterized=True)
            v, hist = helpers.create_histogram(self.data["Igr"][i], n_bins=n_bins)
            ax[0].semilogy(v, hist/max_g, color=b[i], linewidth=1, label=f"{i+1}", rasterized=True)

            # Plot reset histograms for excited state
            v, hist = helpers.create_histogram(self.data["Ier"][i], n_bins=n_bins)
            ax[1].semilogy(v, hist/max_e, color=b[i], linewidth=1, label=f"{i+1}", rasterized=True)

        ax[0].axhline(0.5, color="gray", linestyle="--", linewidth=1)
        ax[1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
        # Helper function to find bin index closest to a value
        def find_bin_closest_to_value(bins, value):
            return np.argmin(np.abs(bins - value))

        # Find indices for excited state level in different histograms
        ind = find_bin_closest_to_value(v, self.data["ie"])
        ind_e = find_bin_closest_to_value(ve, self.data["ie"])
        ind_g = find_bin_closest_to_value(vg, self.data["ie"])

        # Calculate reset performance metrics
        reset_level = hist[ind]
        e_level = histe[ind_e]
        g_level = histg[ind_g]

        # Print reset performance
        print(
            f"Reset is {reset_level/e_level:3g} of e and {reset_level/g_level:3g} of g"
        )

        # Store reset performance metrics
        self.data["reset_e"] = reset_level / e_level
        self.data["reset_g"] = reset_level / g_level

        # Add legend and titles
        ax[0].legend()
        ax[0].set_title("Ground state")
        ax[1].set_title("Excited state")
        plt.show()
        self.save_fig(fig, "_reset_performance")


# ====================================================== #


## 4WM Looping

In [ ]:
params = {'f_p':5217, 'span_f':30, 'expts_f_p':1, 'gain_p':0.04, 'expts_gain_p':50, 'expts_len':1,'shots':5000}
smpd4wmloop1=SMPD4WMPeLoopExperiment(cfg_dict, qi=1,params=params, display=True)#, style="fine")
#params = {'f_p':5220, 'span_f':30, 'expts_f_p':30, 'gain_p':0.04, 'expts_gain_p':1, 'expts_len':1,'shots':10000, 'do_buffer':False}
#smpd4wmloop0=SMPD4WMPeLoopExperiment(cfg_dict, qi=1,params=params, display=True)#, style="fine")

In [ ]:
from pathlib import Path

blue = "#4053d3"
red = "#b51d14"
int_rgain = True


class SMPD4WMPeLoopExperiment(QickExperiment):
    """
    A class for optimizing single-shot readout experiments by sweeping frequency, gain, and readout length.

    This experiment iterates through a parameter space to find the optimal
    combination of readout frequency, gain, and length that maximizes readout fidelity.

    The parameters for this experiment can be configured via the `params` dictionary.
    If a parameter is not provided, a default value will be used.

    Default Parameters
    ------------------
    - `span_f`: Readout frequency span, defaults to `0.8 * kappa` from the device config.
    - `expts_f`: Number of frequency points, defaults to 5.
    - `expts_gain`: Number of gain points, defaults to 5.
    - `expts_len`: Number of readout length points, defaults to 5.
    - `shots`: Number of shots per measurement, defaults to 10000.
    - `check_f`: Boolean to check the f-state, defaults to `False`.
    - `qubit`: Qubit index, defaults to the one specified in `qi`.
    - `save_data`: Boolean to save the raw data, defaults to `True`.
    - `qubit_chan`: Readout channel, defaults to the one from the hardware config.

    The starting points for frequency, gain, and length are determined based on the
    device configuration and the number of experiment points. If `expts_f`, `expts_gain`,
    or `expts_len` is 1, the starting value is taken directly from the device config.
    Otherwise, it is calculated to center the sweep around the config value.
    """

    def __init__(
        self,
        cfg_dict,
        qi=0,
        go=True,
        params={},
        prefix=None,
        fname=None,
        progress=True,
        style="",
        disp_kwargs=None,
        min_r2=None,
        max_err=None,
        display=True,
        print=False,
        check_params=True,
    ):

        if prefix is None:
            prefix = f"single_shot_opt_qubit_{qi}"

        super().__init__(cfg_dict=cfg_dict, prefix=prefix, progress=progress)
        self.im = cfg_dict["im"]
        self.soccfg = cfg_dict["soc"]
        self.config_file = cfg_dict["cfg_file"]
        self.cfg_dict = cfg_dict

        params_def = {
            "f_p": self.cfg.device.readout.frequency[0] + self.cfg.device.qubit.f_ge[0] - self.cfg.device.readout.frequency[1],
            "gain_p": 0.1,
            "t_p": 10,
            "f_b": self.cfg.device.readout.frequency[0],
            "gain_b": self.cfg.device.readout.gain[0],
            #"t_b": 10,
            "span_f": 50,
            "expts_f_p": 5,
            "expts_gain_p": 5,
            "expts_t_p": 5,
            "expts_len":1,
            "shots": 10000,
            "check_f": False,
            "do_buffer":True,
            "qubit": [qi],
            "save_data": True,
            "qubit_chan": self.cfg.hw.soc.adcs.readout.ch[qi],
        }
        params_def["t_b"] = params_def["t_p"]
        params = {**params_def, **params}

        # Start vals
        if params["expts_f_p"] == 1:
            params_def["start_f"] = params["f_p"]
        else:
            params_def["start_f"] = (
                params["f_p"] - 0.5 * params["span_f"]
            )

        if params["expts_gain_p"] == 1:
            params_def["start_gain"] = params["gain_p"]
            params_def["span_gain"] = 0
        else:
            if style == "fine":
                params_def["start_gain"] = params["gain_p"] * 0.8
                params_def["span_gain"] = 0.4 * params["gain_p"]
            else:
                params_def["start_gain"] = params["gain_p"] * 0.01
                params_def["span_gain"] = 5 * params["gain_p"]
        
        if params["expts_len"] == 1:
            params_def["start_len"] = params["t_p"]
        else:
            if style == "fine":
                params_def["start_len"] = (
                    params["t_p"] * 0.8
                )
                params_def["span_len"] = (
                    0.4 * params["t_p"]
                )
            else:
                params_def["start_len"] = (
                    params["t_p"] * 0.3
                )
                params_def["span_len"] = (
                    1.8 * params["t_p"]
                )
        
        params = {**params_def, **params}
        if params["expts_f_p"] == 1:
            params_def["step_f"] = 0
        else:
            params_def["step_f"] = params["span_f"] / (params["expts_f_p"] - 1)

        if params["expts_gain_p"] == 1:
            params_def["step_gain"] = 0
            params_def["span_gain"] = 0
        else:
            params_def["step_gain"] = params["span_gain"] / (params["expts_gain_p"] - 1)

        
        if params["expts_len"] == 1:
            params_def["step_len"] = 0
        else:
            params_def["step_len"] = params["span_len"] / (params["expts_len"] - 1)
        
        if params["span_gain"] + params["start_gain"] > self.cfg.device.qubit.max_gain:
            params_def["span_gain"] = (
                self.cfg.device.qubit.max_gain - params["start_gain"]
            )
        self.cfg.expt = {**params_def, **params}

        # Check for unexpected parameters
        super().check_params(params)
        if print:
            super().print()
            go = False
        if go:
            self.go(analyze=False, display=False, progress=False, save=True)
            self.analyze()
            self.display()

    def acquire(self, progress=False, debug=False):
        fpts = self.cfg.expt["start_f"] + self.cfg.expt["step_f"] * np.arange(
            self.cfg.expt["expts_f_p"]
        )

        max_gain = self.cfg.expt["start_gain"] + self.cfg.expt["step_gain"] * (
            self.cfg.expt["expts_gain_p"] - 1
        )
        if max_gain > self.cfg.device.qubit.max_gain:
            self.cfg.expt["step_gain"] = (
                self.cfg.device.qubit.max_gain - self.cfg.expt["start_gain"]
            ) / (self.cfg.expt["expts_gain_p"] - 1)
        gainpts = self.cfg.expt["start_gain"] + self.cfg.expt["step_gain"] * np.arange(
            self.cfg.expt["expts_gain_p"]
        )

        lenpts = self.cfg.expt["start_len"] + self.cfg.expt["step_len"] * np.arange(
        self.cfg.expt["expts_len"]
        )

        if "save_data" not in self.cfg.expt:
            self.cfg.expt.save_data = False

        fid = np.zeros(shape=(len(fpts), len(gainpts), len(lenpts)))
        threshold = np.zeros(shape=(len(fpts), len(gainpts), len(lenpts)))
        angle = np.zeros(shape=(len(fpts), len(gainpts), len(lenpts)))
        tm = np.zeros(shape=(len(fpts), len(gainpts), len(lenpts)))
        sigma = np.zeros(shape=(len(fpts), len(gainpts), len(lenpts)))
        if "check_f" not in self.cfg.expt:
            check_f = False
        else:
            check_f = self.cfg.expt.check_f
        Ig, Ie, Qg, Qe = [], [], [], []
        if check_f:
            If, Qf = [], []
        gprog = False
        fprog = False
        lprog = False
        if len(fpts) > 1:
            fprog = True
        else:
            fprog = False
            if len(gainpts) > 1:
                gprog = True
            else:
                gprog = False
                if len(lenpts) > 1:
                    lprog = True
                else:
                    lprog = False

        for f_ind, f in enumerate(tqdm(fpts, disable=not fprog)):
            Ig.append([])
            Ie.append([])
            Qg.append([])
            Qe.append([])
            if check_f:
                If.append([])
                Qf.append([])
            for g_ind, gain in enumerate(tqdm(gainpts, disable=not gprog)):
                Ig[-1].append([])
                Ie[-1].append([])
                Qg[-1].append([])
                Qe[-1].append([])
                if check_f:
                    If[-1].append([])
                    Qf[-1].append([])
                for l_ind, l in enumerate(tqdm(lenpts, disable=not lprog)):
                    
                    shot = SMPD4WMHistogramExperiment(
                        self.cfg_dict,
                        go=False,
                        progress=False,
                        qi=self.cfg.expt.qubit[0],
                        params=dict(
                            #frequency=f,
                            #gain=gain,
                            #readout_length=l,
                            #reps=1,
                            do_buffer=self.cfg.expt.do_buffer,
                            check_e=True,
                            check_f=check_f,
                            shots=self.cfg.expt.shots,
                            save_data=self.cfg.expt.save_data,
                            qubit_chan=self.cfg.expt.qubit_chan,
                            gain_b=self.cfg.expt.gain_b,
                            f_b=self.cfg.expt.f_b,
                            gain_p=gain,
                            f_p=f,
                            #t_p=self.cfg.expt.t_p,
                            t_p = l,
                            t_b = l,
                        ),
                    )
                    #smpd4wm=SMPD4WMHistogramExperiment(cfg_dict, qi=1,params={'gain_b':0.003, 'gain_p':0.3392, 'f_p':5211.05, 't_p':10, 't_b':10, 'shots':10000})
                    # shot.cfg = self.cfg

                    shot.go(analyze=False, display=False, progress=progress, save=False)
                    Ig[-1][-1].append(shot.data["Ig"])
                    Ie[-1][-1].append(shot.data["Ie"])
                    Qg[-1][-1].append(shot.data["Qg"])
                    Qe[-1][-1].append(shot.data["Qe"])
                    if check_f:
                        If[-1][-1].append(shot.data["If"])
                        Qf[-1][-1].append(shot.data["Qf"])
                    results = shot.analyze(verbose=False)
                    fid[f_ind, g_ind, l_ind] = (
                        results["fids"][0] if not check_f else results["fids"][1]
                    )
                    threshold[f_ind, g_ind, l_ind] = (
                        results["thresholds"][0]
                        if not check_f
                        else results["thresholds"][1]
                    )
                    try:
                        tm[f_ind, g_ind, l_ind] = results["tm"]
                        sigma[f_ind, g_ind, l_ind] = results["sigma"]
                    except:
                        pass
                    angle[f_ind, g_ind, l_ind] = results["angle"]
                    # print(f'freq: {f}, gain: {gain}, len: {l}')
                    # print(f'\tfid ge [%]: {100*results["fids"][0]}')
                    # if check_f:
                    #     print(f'\tfid gf [%]: {100*results["fids"][1]:.3f}')

        if check_f:
            self.data["If"] = np.array(If)
            self.data["Qf"] = np.array(Qf)
        if self.cfg.expt.save_data:
            self.data = dict(
                fpts=fpts,
                gainpts=gainpts,
                lenpts=lenpts,
                fid=fid,
                threshold=threshold,
                angle=angle,
                Ig=Ig,
                Ie=Ie,
                Qg=Qg,
                Qe=Qe,
                tm=tm,
                sigma=sigma,
            )
            if check_f:
                self.data["If"] = If
                self.data["Qf"] = Qf
        else:
            self.data = dict(
                fpts=fpts,
                gainpts=gainpts,
                lenpts=lenpts,
                fid=fid,
                threshold=threshold,
                angle=angle,
                tm=tm,
                sigma=sigma,
            )

        for key in self.data.keys():
            self.data[key] = np.array(self.data[key])
        return self.data

    def analyze(self, data=None, fit=True, low_gain=True, **kwargs):
        if data == None:
            data = self.data
        fid = data["fid"]
        fpts = data["fpts"]
        gainpts = data["gainpts"]
        lenpts = data["lenpts"]

        imax = np.unravel_index(np.argmax(fid), shape=fid.shape)
        perc_fid = 0.95
        max_fid = np.max(fid)
        print(f"Max fidelity {100*max_fid:.3f} %")

        print(
            f"Optimal params: \n Freq (MHz) {fpts[imax[0]]:.3f} \n Gain (DAC units) {gainpts[imax[1]]:.3f} \n Readout length (us) {lenpts[imax[2]]:.3f}"
        )
        self.do_more = self.check_edges()
        if low_gain and not self.do_more:
            min_accept = max_fid * perc_fid

            # Find values above threshold
            above_threshold = fid >= min_accept

            # For each row, get first index that's above threshold
            freq_indices = np.where(above_threshold)[0]
            gain_indices = np.where(above_threshold)[1]
            time_indices = np.where(above_threshold)[2]
            min_ind = gain_indices + time_indices
            a = np.argmin(min_ind)

            # Get the first occurrence
            imax = (freq_indices[a], gain_indices[a], time_indices[a])
            print(f"Set fidelity: {100*fid[imax]:.3f} %")
            print(
                f"Set params: \n Freq (MHz) {fpts[imax[0]]:.3f} \n Gain (DAC units) {gainpts[imax[1]]:.3f} \n Readout length (us) {lenpts[imax[2]]:.3f}"
            )

        self.data["freq"] = fpts[imax[0]]
        self.data["gain"] = gainpts[imax[1]]
        self.data["length"] = lenpts[imax[2]]

        if self.data["gain"] == 1:  # change to max_gain
            self.do_more = False

        return imax

    def display(self, data=None, plot_pars=False, **kwargs):
        if data is None:
            data = self.data

        fid = data["fid"]

        fpts = data["fpts"]  # outer sweep, index 0
        gainpts = data["gainpts"]  # middle sweep, index 1
        lenpts = data["lenpts"]  # inner sweep, index 2
        ndims = 0
        npts = []
        inds = []
        sweep_var = []
        labs = ["Freq. (MHz)", "Gain", "Readout Length ($\mu$s)"]
        if len(fpts) > 1:
            ndims += 1
            sweep_var.append("fpts")
            npts.append(len(fpts))
            inds.append(0)
        if len(gainpts) > 1:
            ndims += 1
            sweep_var.append("gainpts")
            npts.append(len(gainpts))
            inds.append(1)
        if len(lenpts) > 1:
            ndims += 1
            sweep_var.append("lenpts")
            npts.append(len(lenpts))
            inds.append(2)

        def smart_ax(n):
            row = int(np.ceil(n / 5))
            if n < 5:
                col = n
            else:
                col = 5
            return row, col

        if self.cfg.expt.do_buffer:
            title = f"SMPD 4WM looping pump. t_b = t_p, gain_b = {self.cfg.expt.gain_b}"
        else:
            title = f"SMPD 4WM looping pump. buffer off"

        def return_dim(data, dim, i):
            if len(dim) == 1:
                if dim[0] == 0:
                    return data[i, :, :].reshape(-1)
                elif dim[0] == 1:
                    return data[:, i, :].reshape(-1)
                elif dim[0] == 2:
                    return data[:, :, i].reshape(-1)
            elif len(dim) == 2:
                if dim == [0, 1]:
                    return data[i[0], i[1], :].reshape(-1)
                if dim == [0, 2]:
                    return data[i[0], :, i[1]].reshape(-1)
                if dim == [1, 2]:
                    return data[:, i[0], i[1]].reshape(-1)

        m = 0.5
        imname = Path(self.fname).stem
        folder = Path(self.path)
        '''
        if ndims == 1:
            row, col = smart_ax(npts[0])
            fig, ax = plt.subplots(row, col, figsize=(col * 3, row * 3))
            ax = ax.flatten()
            for i in range(npts[0]):

                ax[i].plot(
                    return_dim(self.data["Ig"], inds, i),
                    return_dim(self.data["Qg"], inds, i),
                    ".",
                    color=blue,
                    alpha=0.2,
                    markersize=m,
                    rasterized=True,
                )
                ax[i].plot(
                    return_dim(self.data["Ie"], inds, i),
                    return_dim(self.data["Qe"], inds, i),
                    ".",
                    color=red,
                    alpha=0.2,
                    markersize=m,
                    rasterized=True,
                )

                ax[i].set_title(f"{labs[inds[0]]} {data[sweep_var[0]][i]:.2f}")
            fig.savefig(folder / "images" / f"{imname}_raw.png")

        elif ndims == 2:
            fig, ax = plt.subplots(npts[0], npts[1], figsize=(npts[1] * 3, npts[0] * 3))

            for i in range(npts[0]):
                for j in range(npts[1]):
                    ax[i, j].plot(
                        return_dim(self.data["Ig"], inds, [i, j]),
                        return_dim(self.data["Qg"], inds, [i, j]),
                        ".",
                        color=blue,
                        alpha=0.2,
                        markersize=m,
                        rasterized=True,
                    )
                    ax[i, j].plot(
                        return_dim(self.data["Ie"], inds, [i, j]),
                        return_dim(self.data["Qe"], inds, [i, j]),
                        ".",
                        color=red,
                        alpha=0.2,
                        markersize=m,
                        rasterized=True,
                    )

                    if i == npts[0] - 1:
                        ax[i, j].set_xlabel(np.round(self.data[sweep_var[1]][j], 2))
                    if j == 0:
                        ax[i, j].set_ylabel(np.round(self.data[sweep_var[0]][i], 2))
            plt.figtext(0.5, 0.0, labs[inds[1]], horizontalalignment="center")
            plt.figtext(
                0.0, 0.5, labs[inds[0]], verticalalignment="center", rotation="vertical"
            )
            fig.savefig(folder / "images" / f"{imname}_raw.png")
        else:
            for k in range(npts[2]):
                fig, ax = plt.subplots(
                    npts[0], npts[1], figsize=(npts[1] * 3, npts[0] * 3)
                )
                for i in range(npts[0]):
                    for j in range(npts[1]):
                        ax[i, j].plot(
                            self.data["Ig"][i, j, k, :],
                            self.data["Qg"][i, j, k],
                            ".",
                            color=blue,
                            alpha=0.2,
                            markersize=m,
                            rasterized=True,
                        )
                        ax[i, j].plot(
                            self.data["Ie"][i, j, k, :],
                            self.data["Qe"][i, j, k],
                            ".",
                            color=red,
                            alpha=0.2,
                            markersize=m,
                            rasterized=True,
                        )
                        if i == npts[0] - 1:
                            ax[i, j].set_xlabel(np.round(self.data[sweep_var[1]][j], 2))
                        if j == 0:
                            ax[i, j].set_ylabel(np.round(self.data[sweep_var[0]][i], 2))

                # Add figure title with constant parameter value
                const_param_value = data[sweep_var[2]][k]
                fig.suptitle(f"{title}, {labs[inds[2]]} = {const_param_value:.2f}")

                # Add figure-level axis labels
                plt.figtext(0.5, 0.0, labs[inds[1]], horizontalalignment="center")
                plt.figtext(
                    0.0, 0.5, labs[inds[0]], verticalalignment="center", rotation="vertical"
                )

                fig.tight_layout()
                fig.savefig(folder / "images" / f"{imname}_raw_{k}.png")

        if self.cfg.expt.do_buffer:
            title = f"SMPD 4WM looping pump. t_b = t_p, gain_b = {self.cfg.expt.gain_b}"
        else:
            title = f"SMPD 4WM looping pump. buffer off"
        '''

        # Summary plots
        if ndims == 1:
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.plot(data[sweep_var[0]], 100 * fid.squeeze(), "o-", color=blue)
            ax.set_xlabel(labs[inds[0]])
            ax.set_ylabel(r"$P_e$ [%]")
            ax.set_title(title)
            fig.tight_layout()
            fig.savefig(folder / "images" / (imname + ".png"))
            plt.show()

        elif ndims == 2:
            fig, ax = plt.subplots(figsize=(7, 5))
            fid_2d = fid.squeeze()
            pcm = ax.pcolormesh(data[sweep_var[0]], data[sweep_var[1]], 100 * fid_2d.T, shading="auto")
            ax.set_xlabel(labs[inds[0]])
            ax.set_ylabel(labs[inds[1]])
            fig.colorbar(pcm, ax=ax, label=r"$P_e$ [%]")
            ax.set_title(title)
            fig.tight_layout()
            fig.savefig(folder / "images" / (imname + ".png"))
            plt.show()

        else:  # ndims == 3
            for k in range(npts[2]):
                fig, ax = plt.subplots(figsize=(7, 5))
                fid_2d = fid[:, :, k]
                pcm = ax.pcolormesh(data[sweep_var[0]], data[sweep_var[1]], 100 * fid_2d.T, shading="auto")
                ax.set_xlabel(labs[inds[0]])
                ax.set_ylabel(labs[inds[1]])
                fig.colorbar(pcm, ax=ax, label=r"$P_e$ [%]")
                ax.set_title(f"{title}\\\\n{labs[inds[2]]} = {data[sweep_var[2]][k]:.3f}")
                fig.tight_layout()
                fig.savefig(folder / "images" / f"{imname}_{k}.png")
                plt.show()


    def check_edges(self):
        do_more = False
        fid = self.data["fid"]
        fid_expts = fid.shape
        if all(dim % 2 != 0 for dim in fid_expts):
            old_fid = fid[(fid_expts[0] // 2), (fid_expts[1] // 2), (fid_expts[2] // 2)]
            max_fid = np.max(fid)
            if (max_fid - old_fid) / old_fid > 0.1:
                print("Fidelity is not maximized at the center of the sweep.")
                max_indices = np.unravel_index(np.argmax(fid), fid.shape)
                print(f"Max fidelity found at indices: {max_indices}")
                if (
                    max_indices[1] == 0
                    or max_indices[1] == fid_expts[1] - 1
                    or max_indices[2] == 0
                    or max_indices[2] == fid_expts[2] - 1
                ):
                    do_more = True
        else:
            print("Not all elements in fid_expts are odd.")
        return do_more

    def update(self, verbose=True):
        qi = self.cfg.expt.qubit[0]
        cfg_file= self.config_file
        config.update_readout(cfg_file, "gain", self.data["gain"], qi, verbose=verbose)
        config.update_readout(
            cfg_file, "readout_length", self.data["length"], qi, verbose=verbose
        )
        config.update_readout(
            cfg_file, "frequency", self.data["freq"], qi, verbose=verbose
        )


In [ ]:
# fit the 4wm 
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(smpd4wmloop1.data['gainpts'], 100 * smpd4wmloop1.data['fid'].squeeze(), "o-", color=blue)
ax.set_xlabel('pump amplitude (arb)')
ax.set_ylabel(r"$P_e$ [%]")
ax.set_title('cavity cooperativity')

xlims = ax.get_xlim()

x = np.linspace(0,xlims[1])

x_ind = np.argmax(100 * smpd4wmloop1.data['fid'].squeeze())
scale = x[x_ind]*1.8

y = 4*(x/scale)**2/(1+(0.08/2)+(x/scale)**2)**2


plt.plot(x,y*100 * np.max(smpd4wmloop1.data['fid'].squeeze()),label=r'$\eta_\mathrm{4wm} = \frac{4\mathcal{C}}{(1+\mathcal{C})^2}$' +'\n'+r'$\mathcal{C}\propto (\mathrm{pump\;amplitude})^2$')

plt.legend()
fig.tight_layout()
fig.savefig("C:\\_Data\\SMPD\\2026_03_06\\images\\summary\\4wm_fit.png")
plt.show()




In [ ]:
data.data.keys()

In [ ]:
params={'span':50,'expts':100,'reps':500,'length_p':30,'start': 5150, 'frequency_b': 6917.3, 'gain_b':0.02,'length_b':30,'sep_readout':True,'start_gain':0.001,'step_gain':0.005,'expts_gain':160,'log':False,'do_buffer':True}
data = meas.QubitSpecShotPower(cfg_dict, qi=1,params=params)

plt.pcolormesh(data.data['xpts'], data.data['ypts'], data.data['p_e'], shading='auto')
plt.xlabel('Gain')
plt.ylabel('Frequency')
plt.title('4WM Spectroscopy')
plt.show()


In [ ]:
# import h5 data
import h5py

# load up most recent power spectroscopy data
files = os.listdir(expt_path + "\\data\\")
file = sorted([f for f in files if f.startswith('qubit_spectroscopy_power')])[-1]  # Get the most recent file
with h5py.File(expt_path + "\\data\\" + file, "r") as f:
    # List all groups
    print("Keys: %s" % f.keys())
    p_e = f['p_e'][:]
    xpts = f['xpts'][:]
    gain_p_pts = f['gain_p_pts'][:]



max_indices = np.argmax(p_e, axis=1)
# plt.plot(xpts[max_indices][10:], gain_p_pts[10:], 'o-', label='Max P(e)', color='red')
# plt.legend()


plt.figure()
max_p_e = np.max(p_e, axis=1)
plt.plot(gain_p_pts[10:], max_p_e[10:], 'o-', label='Max P(e)', color='red')
plt.xlabel("Pump Gain (DAC units)")
plt.ylabel("Max Excited State Probability")
plt.legend()

# frequency at which max p_e occurs vs gain

freq_devs = ( xpts[max_indices][10:])

#fit a parabola to freq_devs vs gain_p_pts
from scipy.optimize import curve_fit
def cubic(x, a, c, a3):
    # force parabola to open upwards and have vertex at 0 by setting b=0 and c=0
    return a3*x**3 + a*x**2 + c
popt, pcov = curve_fit(cubic, gain_p_pts[10:], freq_devs)


plt.figure()
plt.plot(gain_p_pts[10:], freq_devs , 'o-', label='Frequency at Max P(e)', color='green')
plt.plot(gain_p_pts, cubic(gain_p_pts, *popt), label='Cubic Fit', color='orange')
plt.xlabel("Pump Gain (DAC units)")
plt.ylabel("Pump Frequency at Max P(e) (MHz)")
plt.legend()

print(f"Fitted cubic parameters: a={popt[0]:.3e}, c={popt[1]:.3f} MHz, a3={popt[2]:.3e}")

#print out pump frequency along the cubic where gain = 0.3
target_gain = 0.3
target_freq = cubic(target_gain, *popt)
print(f"Pump frequency at gain {target_gain}: {target_freq:.2f} MHz")   

#find nearest pump_frequency points to the best fit parabola at all the gain_p_pts
best_fit_freqs = cubic(gain_p_pts, *popt)

plt.figure()
plt.pcolormesh(xpts, gain_p_pts, p_e, shading='auto')
plt.ylabel("Pump Gain (DAC units)")
plt.xlabel("Pump Frequency (MHz)")
plt.colorbar(label="Excited State Probability")
#plot the best fit cubic on top
plt.plot(best_fit_freqs, gain_p_pts, label='Best Fit Cubic', color='orange')
plt.tight_layout()
plt.savefig(expt_path + "\\images\\summary\\4WM_spec_power.png", dpi=200)


# iterate through the gain_p_pts and find the nearest xpts to the best fit cubic, then plot the p_e vs xpts for those gain_p_pts
plt.figure()
for i, gain in enumerate(gain_p_pts):
    best_freq = cubic(gain, *popt)
    nearest_freq_idx = np.argmin(np.abs(xpts - best_freq))
    # plt.plot(xpts, p_e[i,:], label=f'Gain {gain:.3f}')
    plt.plot(gain, p_e[i, nearest_freq_idx], 'o', color='black')  # mark the nearest point
plt.grid(True)
plt.xlabel("Pump Gain (DAC units)")
plt.title("Excited State Probability at Best Fit Frequency")
plt.ylabel("p(e)")
plt.ylim(0,1)
plt.tight_layout()
plt.savefig(expt_path + "\\images\\summary\\4WM_spec_power_cubic_points.png", dpi=200)

# find the frequency along the cubic where gain = 0.25, then plot the p_e vs gain at that frequency
target_gain = 0.25
target_freq = cubic(target_gain, *popt)
nearest_freq_idx = np.argmin(np.abs(xpts - target_freq))
plt.figure()
plt.plot(gain_p_pts, p_e[:, nearest_freq_idx], 'o-', label=f'P(e) at freq={target_freq:.2f} MHz', color='purple')
plt.title(f'pump gain = {target_gain:.2f}')
plt.xlabel("Pump Gain (DAC units)") 
plt.ylabel("Excited State Probability")
plt.legend()

In [ ]:
np.sqrt(-1*popt[0]/248*(4373.55-popt[1])**2)    

In [ ]:
params={'reps':1000,'gain_p':0.25,'length_p':10,'start': 5175, 'span':18,'expts':300, 'center_freq_b': 6917.3, 'expts_b':300,'gain_b':0.005, 'span_b':1,'length_b':10,'delay_b':0,'sep_readout':True}
data = meas.QubitSpecShotBuffer(cfg_dict, qi=1,params=params)

In [ ]:
# find most recent qubit_spectroscopy_buffer file in the data-taking folder
# Assuming the most recent file is the one with the highest number
file = sorted([f for f in os.listdir(expt_path+'\\data') if f.startswith("qubit_spectroscopy_buffer")])[-1]

with h5py.File(f"{expt_path}\\data\\{file}", "r") as f:
    # List all groups
    print("Keys: %s" % f.keys())
    p_e = f['p_e'][:]
    xpts = f['xpts'][:]
    frequency_b_pts = f['frequency_b_pts'][:]

plt.figure()
plt.pcolormesh(xpts, frequency_b_pts, p_e, shading='auto')
plt.xlabel("Qubit Frequency (MHz)")
plt.ylabel("Buffer Frequency (MHz)")
plt.colorbar(label="Excited State Probability")

# find the pump frequency at which data.data['p_e'] has the broadest bandwidth and plot p_e vs buffer frequency for that pump frequency
# plt.figure()
# plt.plot(np.std(data.data['p_e'], axis=1))
# print(np.argmax(np.std(data.data['p_e'], axis=1)))
# pump_freq = data.data['xpts'][np.argmax(np.std(data.data['p_e'], axis=1))]
pump_freq = 5187
plt.figure()
plt.plot(frequency_b_pts, p_e[:, np.argmin(np.abs(xpts - pump_freq))])
plt.xlabel("Buffer Frequency (MHz)")
plt.ylabel("Excited State Probability")
plt.title(f"Pump Frequency: {pump_freq:.2f} MHz")


In [ ]:
plt.plot(xpts,p_e[np.argmin(np.abs(frequency_b_pts - 6917.6)),:])

p_e_cut = p_e[np.argmin(np.abs(frequency_b_pts - 6917.6)),:]
# list the xpts idcs and frequencies in order of decreasing p_e
sorted_indices = np.argsort(p_e_cut)[::-1]
sorted_freqs = xpts[sorted_indices]
print("Frequencies sorted by p_e:" )
for freq, p in zip(sorted_freqs, p_e_cut[sorted_indices]):
    print(f"Frequency: {freq:.2f} MHz, p_e: {p:.3f}")
    # list index of the frequency in the original xpts array
    print(f"Index in xpts: {np.where(xpts == freq)[0][0]}")

In [ ]:
from matplotlib.colors import Normalize

def cavity_transmission(fb,fp,fb0,fp0,kb,kw,C,flat,A):
    db = (fb - fb0)
    dp = (fp - fp0)
    return A*4*C/abs(1+C-4*db*(db+dp)/kb/kw+2j*db/kb+2j*(db+dp)/kw)**2 + flat

# fit the cavity transmission to the 2D p_e data that depends on buffer and pump frequency to extract the cooperativity C and the flat background and the frequency offsets

# fp0 = 5184
from scipy.optimize import curve_fit
def fit_func(X,fb0,fp0,kb,kw, C, flat, A):
    fb, fp = X # unpack the meshgrid

    return cavity_transmission(fb, fp, fb0, fp0, kb, kw, C, flat, A).flatten()
# prepare the data for fitting
X = np.meshgrid(frequency_b_pts, xpts)

# there are three columns in the data that are clearly bad and should be removed before fitting, find the indices of those columns in X and p_e and set them nan
bad_indices = [76,263]  # example indices of bad columns
for idx in bad_indices:
    p_e[:, idx] = 0.3

norm = Normalize(vmin=0, vmax=1)



plt.figure()
plt.pcolormesh(xpts, frequency_b_pts, p_e, norm=norm,cmap='magma')
plt.xlabel("Qubit Frequency (MHz)")
plt.ylabel("Buffer Frequency (MHz)")
c = plt.colorbar(label="Excited State Probability")
plt.tight_layout()
# make figure name reference the file it comes from
plt.savefig(f"{expt_path}\\images\\{file[:-4]}.png", dpi=200)


y = p_e.flatten()

# provide initial guesses for the fit parameters
initial_guesses = [6917.1, 5184, 0.22, 2, 1, 0.2, 0.8]
lb = [6917, 5184*0.9, initial_guesses[2]*0.1, initial_guesses[3]*0.1, 0, 0, 0]
ub = [6917.5, 5184*1.1, initial_guesses[2]*10, initial_guesses[3]*10, 5, 1, 1]
# perform the curve fitting
popt, pcov = curve_fit(fit_func, X, y, p0=initial_guesses, bounds=(lb, ub))
print(f"Fitted parameters: fb0={popt[0]:.2f} MHz, fp0={popt[1]:.2f} MHz, kb={popt[2]:.3f}, kw={popt[3]:.3f}, C={popt[4]:.3f}, flat={popt[5]:.3f}, A={popt[6]:.3f}")
# plot the fitted function as a pcolormesh
fb_fit = np.linspace(frequency_b_pts.min(), frequency_b_pts.max(), 100)
fp_fit = np.linspace(xpts.min(), xpts.max(), 100)
FB_fit, FP_fit = np.meshgrid(fb_fit, fp_fit)
Z_fit = cavity_transmission(FB_fit, FP_fit, *popt[:1],*popt[1:3], *popt[3:7]).reshape(FB_fit.shape)

plt.figure()
# plot the colormesh of the fit but use the same colorbar as from the previous figure
plt.pcolormesh(FP_fit, FB_fit, Z_fit, shading='auto',norm=norm,cmap='magma')
plt.xlabel("Pump Frequency (MHz)")
plt.ylabel("Buffer Frequency (MHz)")
plt.colorbar(label="Fitted Excited State Probability")
plt.title("Fitted Cavity Transmission")
plt.plot([],[],label='C = {:.2f}\nflat = {:.2f}\nA = {:.2f}'.format(popt[4], popt[5], popt[6]), ls='',marker='')
plt.legend()
plt.tight_layout()
plt.savefig(f"{expt_path}\\images\\{file[:-4]}_fit.png", dpi=200)


plt.figure()
#plot an identical colormesh except for C = 1
Z_fit_C1 = cavity_transmission(FB_fit, FP_fit, *popt[:1], *popt[1:3], 1, *popt[4:7]).reshape(FB_fit.shape)
plt.pcolormesh(FP_fit, FB_fit, Z_fit_C1, shading='auto', norm=norm,cmap='magma')
plt.xlabel("Pump Frequency (MHz)")
plt.ylabel("Buffer Frequency (MHz)")
plt.colorbar(label="Fitted Excited State Probability")
plt.title("Fitted Cavity Transmission with C=1")
plt.tight_layout()
plt.savefig(f"{expt_path}\\images\\{file[:-4]}_fit_C1.png", dpi=200)




In [ ]:
params={'span':0,'expts':1,'gain_p':0.3,'reps':2000,'length_p':10,'start': 5187, 'frequency_b': 6917.3, 'gain_b':0.01,'sep_readout':True,'save_shots':True,'do_buffer':True}
data = meas.QubitSpecShot(cfg_dict, qi=1,params=params,display=False)

params={'span':0,'expts':1,'gain_p':0.3,'reps':2000,'length_p':10,'start': 5187, 'frequency_b': 6917.3, 'gain_b':0.01,'sep_readout':True,'save_shots':True,'do_buffer':False}
data_dark = meas.QubitSpecShot(cfg_dict, qi=1,params=params,display=False)

params={'span':0,'expts':1,'gain_p':0.3,'reps':2000,'length_p':10,'start': 5187, 'frequency_b': 6917.3, 'gain_b':0.01,'sep_readout':True,'save_shots':True,'do_buffer':False,'do_pump':False}
data_nopump = meas.QubitSpecShot(cfg_dict, qi=1,params=params,display=False)
print(data.data['p_e'])
print(data_dark.data['p_e'])
print(data_nopump.data['p_e'])

In [ ]:
hist = plt.hist(data.data['shots'][0,0,:,0,0],bins=50,alpha=0.5)
hist_dark = plt.hist(data_dark.data['shots'][0,0,:,0,0],bins=50,alpha=0.5)
hist_nopump = plt.hist(data_nopump.data['shots'][0,0,:,0,0],bins=50,alpha=0.5)

## 4WM gainb and tb sweep

In [ ]:
cubic(0.25,*popt)

In [ ]:
gain_p = 0.25
frequency_p = cubic(0.25,*popt)

gainb_sweep = meas.FWM_gainb_sweep(cfg_dict, qi=1, params={'gain_p':gain_p,'frequency_p':frequency_p,'length_p':50,'start':0.001,'span':0.1,'expts':100,'frequency_b':6917.3,'length_b':15,'reps':10000}, display=True)

plt.plot(gainb_sweep.data['xpts'],gainb_sweep.data['p_e'])
plt.xlabel('buffer gain')
plt.ylabel(r'p_e')

In [ ]:
n_photon_conversion

In [ ]:
# compute probability of 0 for a poisson distribution with mean given by the xpts of the gainb_sweep data
from scipy.stats import poisson
n_photons = gainb_sweep.data['xpts']**2*n_photon_conversion
poisson_zero = poisson.pmf(0, n_photons)
plt.plot(n_photons, 1-poisson_zero, label='1-P(0)', color='orange')
plt.legend()


In [ ]:
plt.plot(1-poisson_zero, gainb_sweep.data['p_e'])
plt.xlabel('1-P(0)')
plt.ylabel(r'p_e')

# fit for the slope of p_e vs 1-P(0) at low gain
low_gain_mask = (1-poisson_zero < 0.15) & (1-poisson_zero > 0.02)
p_e_fit = np.polyfit(1-poisson_zero[low_gain_mask], gainb_sweep.data['p_e'][low_gain_mask], 1)
efficiency = p_e_fit[0]
print(f"Estimated efficiency: {efficiency:.3f}")


In [ ]:
tb_sweep = meas.FWM_tb_sweep(cfg_dict, qi=1, params={'gain_p':0.25,'frequency_p':5183.98,'length_p':10,'start':1,'span':9,'expts':200,'gain_b':0.01,'frequency_b':6917.3,'reps':2000}, display=True)

plt.plot(tb_sweep.data['xpts'],tb_sweep.data['p_e'])

In [ ]:
np.arange(0.1,0.8,0.025)

In [ ]:
gainb_tb_sweeps = []

for gain_p in np.arange(0.1,0.8,0.025):
    frequency_p = cubic(0.25,*popt)

    gainb_tb_sweep = meas.FWM_gainb_tb_sweep(cfg_dict, qi=1, params={'gain_p':gain_p,'frequency_p':frequency_p,'length_p':50,'start':0.001,'span':0.08,'expts':100,'frequency_b':6917.3,'reps':1000,'length_b_start':0.1,'length_b_end':50,'length_b_expts':100}, display=True)

    plt.pcolormesh(gainb_tb_sweep.data['xpts'], gainb_tb_sweep.data['ypts'], gainb_tb_sweep.data['p_e'])
    plt.xlabel('buffer gain')
    plt.ylabel('buffer length (us)')
    plt.colorbar(label=r'$p_e$')
    plt.show()

    gainb_tb_sweeps.append(gainb_tb_sweep)

In [ ]:
import os

In [ ]:
# loop through files in C:\_Data\SMPD\2026_04_11_v2_calibration\data
gainb_tb_sweeps = []
for file in os.listdir('C:\\_Data\\SMPD\\2026_04_11_v2_calibration\\data'):
    if 'lengthb_gainb' in file and not '00001' in file and not '00000' in file:
        gainb_tb_sweeps.append(file)
print(gainb_tb_sweeps)
# gainb_tb_sweep = gainb_tb_sweeps[5]

In [ ]:
import h5py
with h5py.File('C:\\_Data\\SMPD\\2026_04_11_v2_calibration\\data\\' +gainb_tb_sweeps[0],'r') as f:
    print(f.keys())
    print(f['time'][:])

In [ ]:
# for each buffer length, fit for the slope of p_e vs gain_b between 0.005 and 0.010 and plot it as a function of buffer length

pump_amplitudes = np.arange(0.1,0.8,0.025)
peak_efficiencies = []
for i in range(len(gainb_tb_sweeps)):
    print('===============================')
    print('doing pump amplitude: {:.3f}'.format(pump_amplitudes[i]))

    gainb_tb_sweep = gainb_tb_sweeps[i]
    fig,ax = plt.subplots(figsize=(6,4),nrows=1,ncols=2)
    slopes = []
    highlighted_lengths = [1,2,5,10,20,50]
    idcs = []
    for length in highlighted_lengths:
        #find the right index in gainb_tb_sweep.data['ypts'] that corresponds to this length
        idx = np.argmin(np.abs(gainb_tb_sweep.data['ypts'] - length))
        idcs.append(idx)


    for i in range(gainb_tb_sweep.data['ypts'].shape[0]):
        p_e_slice = gainb_tb_sweep.data['p_e'][i,:]
        # convert gain_b to 1-P(0) using the same conversion as before
        gain_b_slice = gainb_tb_sweep.data['xpts']

        # plot p_e_slice vs gain_b_slice for some characteristic buffer length values to see how the slope changes with buffer length
        if i in idcs:
            ax[0].plot(gain_b_slice, p_e_slice)
            # draw a line at one photon gain for reference
            one_photon_gain = np.sqrt(1/n_photon_conversion)
            ax[0].axvline(one_photon_gain, color='red', linestyle='--')
            ax[0].set_xlabel('buffer gain')
            ax[0].set_ylabel('p_e')

        n_photons = gain_b_slice**2 * n_photon_conversion
        poisson_zero = poisson.pmf(0, n_photons)
        gain_b_slice = 1 - poisson_zero

        if i in idcs:
            ax[1].plot(gain_b_slice, p_e_slice, label=f'buffer length: {gainb_tb_sweep.data["ypts"][i]:.1f} us')
            # draw a line at one photon gain for reference


        # find indices corresponding to 1-P(0) of 0.05 and 0.2
        idx_05 = np.argmin(np.abs(gain_b_slice - 0.05))
        idx_02 = np.argmin(np.abs(gain_b_slice - 0.2))


        # fit for slope between these two points
        p_e_slice = p_e_slice[idx_05:idx_02]
        gain_b_slice = gain_b_slice[idx_05:idx_02]
        # fit a linear function to the data
        coeffs = np.polyfit(gain_b_slice, p_e_slice, 1)
        slope = coeffs[0]
        slopes.append(slope)

    one_photon_gain = 1 - poisson.pmf(0, 1)
    ax[1].axvline(one_photon_gain, color='red', linestyle='--',label='one photon gain')
    ax[1].set_xlabel('1-P(0)')

    # put legend to the right of the plot
    plt.legend(bbox_to_anchor=(1.05, 1))

    # fit slope data to a product of two exponentials of the form A*(B-exp(-x/tau1))*(exp(-x/tau2)+C) and plot the fit



    plt.figure()
    plt.plot(gainb_tb_sweep.data['ypts'], slopes)
    try: 
        def fit_func(x, A, B, tau1, tau2, C):
            return A*(B-np.exp(-x/tau1))*(np.exp(-x/tau2)+C)
        from scipy.optimize import curve_fit
        lb = [0,0,0,0,0]
        ub = [100,100,100,100,100]
        popt, pcov = curve_fit(fit_func, gainb_tb_sweep.data['ypts'], slopes, p0=[0.5, 1, 0.2, 20, 0.1])
        plt.plot(gainb_tb_sweep.data['ypts'], fit_func(gainb_tb_sweep.data['ypts'], *popt), label=f'kb={1/popt[2]*(2*np.pi):.2f} MHz, T1={popt[3]:.2f} us', color='orange')
        #plot the individual exponential components of the fit
        plt.plot(gainb_tb_sweep.data['ypts'], popt[0]*(popt[1]-np.exp(-gainb_tb_sweep.data['ypts']/popt[2])), label='kb component', color='green')
        plt.plot(gainb_tb_sweep.data['ypts'], popt[0]*(np.exp(-gainb_tb_sweep.data['ypts']/popt[3])+popt[4]), label='T1 component', color='red')

        #show the functional form of the fit as an equation off to the side of the plot using plt.text
        plt.text(1.1, 0.5, f'$A(B-e^{{-x/\\tau_1}})(e^{{-x/\\tau_2}}+C)$\nA={popt[0]:.2f}\nB={popt[1]:.2f}\n$\\tau_1$={popt[2]:.2f} us\n$\\tau_2$={popt[3]:.2f} us\nC={popt[4]:.2f}', transform=plt.gca().transAxes, fontsize=10, verticalalignment='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

        peak_efficiencies.append(max(fit_func(gainb_tb_sweep.data['ypts'],*popt)))

    except:
        print('error fitting')
        peak_efficiencies.append(None)
        pass
    plt.xlabel('buffer length (us)')
    plt.ylabel('efficiency')
    plt.ylim(-0.05, 1.5)
    plt.legend()
    plt.show()

    print(' ')
    print('  ')


In [ ]:
import FridgeControl

In [ ]:
FridgeControl.setMXCHeater(16)

In [ ]:
heats = np.array([0,16])
temps = np.array([9.5,22.7])

# extrapolate out to heater values at 30 mK, 40 mK, and 50 mK using a quadratic fit to the existing data
coeffs = np.polyfit( temps**2, heats,1)

temps_plot = np.array([10,25,30,40,50])

plt.plot(temps_plot**2, np.polyval(coeffs, temps_plot**2),'o-')
# change x axis tick labels to (temperature)^2 quantities
plt.xticks(temps_plot**2, labels=[f'({temp:.1f} mK)'+r'$^2$' for temp in temps_plot])
plt.grid(True)

temps_plot = np.array([60,70,80,90,100])
plt.figure()
plt.plot(temps_plot**2, np.polyval(coeffs, temps_plot**2),'o-')
# change x axis tick labels to (temperature)^2 quantities
plt.xticks(temps_plot**2, labels=[f'({temp:.1f} mK)'+r'$^2$' for temp in temps_plot])
plt.grid(True)

temps = np.array([25,30,40,50,60,70,80,90,100])
heats = np.polyval(coeffs, temps**2)
print(heats)

In [ ]:


# load up the temperature file and find all the temperature values

with open(expt_path + '\\temperature_log.txt', 'r') as f:
    temperature_values = []
    for line in f:
        if line.startswith("Temperature:"):
            temp = float(line.split(":")[1].split(" K")[0])
            temperature_values.append(temp)
temperature_values = np.array(temperature_values)

# average every three temperature values and make the final array three times so that it has the same length as the number of gainb_tb_sweeps
temperature_values_avg = np.mean(temperature_values.reshape(-1, 3), axis=1
)
# fit a line to heats vs temperature_values_avg**2 and print the coefficients
coeffs = np.polyfit(temperature_values_avg[1:]**2, heats, 1)

print(temperature_values_avg)
plt.figure(figsize=(7,5))
plt.plot(temperature_values_avg[1:]**2,heats,'o')
plt.plot(np.array([0,0.1])**2, np.polyval(coeffs, np.array([0,0.1])**2), label='heater power at 100 mK: {:.3g} uW\ny-intercept: {:.3g} uW'.format(np.polyval(coeffs, 0.1**2), np.polyval(coeffs, 0)), color='orange')

xticks =np.array([0,20,40,60,80,100])/1000
labels =[f'({temp*1000:.1f} mK)'+r'$^2$' for temp in xticks]
labels[0] = '0 mK'
plt.xticks(xticks**2, labels=labels,rotation=30)
plt.ylabel('Heater power (uW)')
plt.xlabel('Temperature')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(expt_path + '\\images\\summary\\heater_power_vs_temperature.png', dpi=200)

In [ ]:
for heat in heats:
    print('setting heater to {:.2f} for target temperature of {:.1f} mK'.format(heat, np.sqrt(heat/coeffs[0]-coeffs[1]/coeffs[0])))

In [ ]:
import time


temps = np.array([30,40,50,60,70,80,90,100])
heats = np.polyval(coeffs, temps**2)
for heat in heats:
    print('setting heater to {:.2f} for target temperature of {:.1f} mK'.format(heat, np.sqrt(heat/coeffs[0]-coeffs[1]/coeffs[0])))
    FridgeControl.setMXCHeater(round(heat,2))
    time.sleep(1800)  # wait 30 minutes for the temperature to stabilize

    measured_temp_1 = bf.get_mxc_temperature()

    tp_sweep = meas.FWM_tp_sweep(cfg_dict, qi=1, params={'gain_p':0.25,'frequency_p':5183.83,'start':1,'stop':100,'expts':40,'reps':40000,'active_reset':False}, display=True)

    measured_temp_2 = bf.get_mxc_temperature()

    plt.plot(tp_sweep.data['xpts'], tp_sweep.data['p_e'])
    plt.xlabel('pump length (us)')
    plt.ylabel(r'p_e')
    plt.title('Pump Length Sweep')

    time.sleep(10) # pasue for 10 seconds between measurements
    t2r = meas.T2Experiment(cfg_dict, qi=qi, max_err=10,params={'reps':10000})

    measured_temp_3 = bf.get_mxc_temperature()

    # save the temperature and filename to a text file
    with open(expt_path + '\\temperature_log.txt', 'a') as f:
        f.write(f"Temperature: {measured_temp_1:.3f} K, Filename: {tp_sweep.fname}\n")
        f.write(f"Temperature: {measured_temp_2:.3f} K, Filename: {t2r.fname}\n")
        f.write(f"Temperature: {measured_temp_3:.3f} K, Filename: {t2r.fname}\n")

    # measure and record temperature
    print(f"MXC: {measured_temp_3*1000:.2f} mK")



In [ ]:
t2r = meas.T2Experiment(cfg_dict, qi=qi, max_err=10,params={'reps':10000})

measured_temp_3 = bf.get_mxc_temperature()

# save the temperature and filename to a text file
with open(expt_path + '\\temperature_log.txt', 'a') as f:
    f.write(f"Temperature: {measured_temp_1:.3f} K, Filename: {tp_sweep.fname}\n")
    f.write(f"Temperature: {measured_temp_2:.3f} K, Filename: {t2r.fname}\n")
    f.write(f"Temperature: {measured_temp_3:.3f} K, Filename: {t2r.fname}\n")

In [ ]:
import h5py
# load most recent tp sweep data
files = os.listdir(expt_path + "\\data\\")
file = sorted([f for f in files if f.startswith('4WM_sweep_tp')])[-1]  # Get the most recent file
with h5py.File(expt_path + "\\data\\" + file, "r") as f:
    # List all groups
    print("Keys: %s" % f.keys())
    p_e = f['p_e'][:]
    xpts = f['xpts'][:]

#fit p_e vs pump length to an exponential saturation curve of the form p_e = A*(1-exp(-x/tau)) + offset
from scipy.optimize import curve_fit
def exp_saturation(x, A, tau, offset):
    return A*(1-np.exp(-x/tau)) + offset

# assign 3x larger error bars for pump lengths below 20 us to account for pulse distortion effects that are not captured by the simple exponential saturation model
popt, pcov = curve_fit(exp_saturation, xpts, p_e, p0=[0.5, 10, 0.1], sigma=np.where(xpts<20, 0.03, 0.01))

# compute the slope at the y-intercept using the fitted parameters
slope_at_zero = popt[0]/popt[1]

# compute uncertainties in the slope at zero using error propagation
A_err, tau_err, offset_err = np.sqrt(np.diag(pcov))
slope_at_zero_err = slope_at_zero * np.sqrt((A_err/popt[0])**2 + (tau_err/popt[1])**2)

# get current temperature
curr_temp = bf.get_mxc_temperature()*1000

plt.figure()
plt.plot(xpts, p_e, 'o-')
ylims =plt.gca().get_ylim()
plt.plot(xpts, exp_saturation(xpts, *popt), label=f'tau={popt[1]:.2f} us', color='orange')
#plot the slope at the y-intercept as a dashed line
plt.plot(xpts, slope_at_zero*xpts + popt[2], label=f'slope at zero = {slope_at_zero*1e6:.3f}/s ± {slope_at_zero_err*1e6:.3f}/s', color='green', linestyle='--')
plt.plot([],[], label=f'Temperature: {curr_temp:.2f} mK', ls='', marker='')
#show the functional form of the fit as an equation off to the side of the plot using plt.text
plt.text(1.1 , 0.5, f'$p_e = A(1-e^{{-x/\\tau}}) + offset$\nA={popt[0]:.2f}\n$\\tau$={popt[1]:.2f} us\noffset={popt[2]:.2f}', transform=plt.gca().transAxes, fontsize=10, verticalalignment='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
plt.ylim(ylims)
plt.xlabel("Pump Length (us)")
plt.ylabel("Excited State Probability")
plt.title("Pump Length Sweep")
plt.legend()
plt.savefig(f"{expt_path}\\images\\summary\\4WM_tp_sweep_fit_{file}.png", dpi=200)


# Active reset

### trying unconditional active reset

In [ ]:
f0g1_transition = 4370.941 + 4123.3209 - 7742.1226

meas.QubitSpec(cfg_dict, qi=1, 
               params={'checkEF':True, 
                       'checkFG':True,
                       'span':80,
                       'expts':100,
                       'gain':1,
                       'reps':10000,
                       'length':4,
                       'start': f0g1_transition-40,
                       'sep_readout':True}, display=True)

In [ ]:
qubit_list=[1]
#qubit_list=np.arange(6)

f0g1_transition = 4370.941 + 4123.3209 - 7742.1226
for qi in qubit_list:
    # params={'start':3700,'span':200,'expts':300}
    #qspec_pow = meas.QubitSpecPower(cfg_dict, qi=qi, style='', params=params)
    qspec_pow = meas.QubitSpecPower(cfg_dict, qi=qi, style='', 
                                    params={'start':f0g1_transition,
                                            'span':3,
                                            'rng': 200,
                                            'reps':50,
                                            'rounds':5,
                                            'expts':200,
                                            'expts_gain':20, 
                                            'max_gain':0.1, 
                                            'length':1,'sep_readout':True,
                                            'checkEF':True,
                                            'checkFG':True})

### Check active reset at standard threshold

Setup reset uses the calibrated angle and runs active reset process at end, but also uses usual final_delay so that it's ok if reset not working

In [ ]:
qubit_list = np.arange(3)
qubit_list=[1]
for qi in qubit_list:
    shot = meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':30000,'active_reset':True, 'setup_reset':True})
    shot.check_reset()
    #config.update_readout(cfg_path, 'reset_e', shot.data['reset_e'], qi)
    #config.update_readout(cfg_path, 'reset_g', shot.data['reset_g'], qi)


### Try sweeping gain/time to see where active reset works

In [ ]:
qubit_list = [3]
length =[10, 15, 20, 25]
gain = [0.05]
for qi in qubit_list:
    for l in length:
        for g in gain:
            shot = meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':10000,'readout_length':l, 'gain':g})
            shot.update()
            shot = meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':20000,'active_reset':True, 'setup_reset':True, 'readout_length':l, 'gain':g})
            shot.check_reset()

## Don't do reset, but measure repeatedly 

In [ ]:
qi=4
shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'remeas':True, 'shots':40000})
shot.check_reset()

## Plot fidelity vs v threshold for active reset

In [ ]:
qi=1
npts = 31
fids = []

auto_cfg = config.load(cfg_path)
threshold = auto_cfg['device']['readout']['threshold'][qi]
sigma = auto_cfg['device']['readout']['sigma'][qi]
rng=4*sigma
shot_list=[]
thresh = np.linspace(threshold-rng/2,threshold+rng/2,npts)
for i, t in enumerate(thresh):
    shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':30000,'threshold_v':t, 'active_reset':True,'setup_reset':False, 'reset':3}, display=False, progress=False)
    fids.append(float(shot.data['fids'][0]))
    shot_list.append(shot)
    if i%6==0:
        print(f'Completed {i}/{len(thresh)}')

from pathlib import Path

fig = plt.figure()
plt.plot(thresh, fids, 'o-')
plt.xlabel('Threshold (ADC units)')
plt.ylabel('Fidelity')
plt.axvline(threshold, color='r', linestyle='--')
plt.legend()
a = np.argmax(fids)

plt.text(0.05, 0.94,
    f'Fidelity: {fids[a]:.4f}\nThreshold: {thresh[a]:.4f}',
    fontsize=12, ha='left', va='top',
    bbox=dict(facecolor='white', boxstyle='round,pad=0.5'),
    transform=plt.gca().transAxes,   
)

file_path = Path(shot.fname)
new_filename = file_path.name.rsplit(".", 1)[0] + "_fidelity_reset.png"
fig.savefig(file_path.parent / "images" / new_filename)
config.update_readout(cfg_dict['cfg_file'], "threshold", thresh[a], qi);

In [ ]:
import seaborn as sns
sns.set_palette('coolwarm', n_colors=len(shot_list)//2+1)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for i,shot in enumerate(shot_list):
    if i%2==0:
        ax[0].plot(shot.data['histg'], label='Ground',  linewidth=1)
        ax[1].plot(shot.data['histe'], label='Excited', linewidth=1)

In [ ]:
# If the threshold is too high, it won't do pi pulse on a lot of excited states, and they will stay excited 
# If the threshold is too low, it will do pi pulses on a lot of ground states 
# If you don't have enough wait time, the resonator will be excited when in the ground state, and even when your threshold is too low, pi pulses on the ground state won't do anything. 
#  

## Test single shot (no wait time between shots)

In [ ]:
qi=0
shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':30000, 'active_reset':True,'setup_reset':False}, display=False)
shot.check_reset()

## Play with final delay

In [ ]:
fidelity=[]
final=[1, 3, 5, 7, 10]

qi=0
for f in final:
    shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':20000, 'active_reset':True,'final_delay':f, 'reset':1}, display=False)   

    fidelity.append(shot.data['fids'][0])

## Compare readouts for using and not using active reset for standard scans

### Rabi

In [ ]:
qi=3
amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi, params={'active_reset':True})
amp_rabi2 = meas.RabiExperiment(cfg_dict,qi=qi, params={'active_reset':False})

plt.figure()
plt.plot(amp_rabi.data['xpts'], amp_rabi.data['avgi'], label='Active Reset')
plt.plot(amp_rabi2.data['xpts'], amp_rabi2.data['avgi'], label='No Active Reset')
plt.legend()
# plt.figure()
# plt.plot(amp_rabi.data['xpts'], amp_rabi.data['avgq'])
# plt.plot(amp_rabi2.data['xpts'], amp_rabi2.data['avgq'])


### T1

In [ ]:
qi=1
t1 = meas.T1Experiment(cfg_dict,qi=qi, params={'active_reset':True})
t12 = meas.T1Experiment(cfg_dict,qi=qi, params={'active_reset':False})

plt.figure()
plt.plot(t1.data['xpts'], t1.data['avgi'], label='Active Reset')
plt.plot(t12.data['xpts'], t12.data['avgi'])
plt.legend()
# plt.figure()
# plt.plot(t1.data['xpts'], t1.data['avgq'])
# plt.plot(t12.data['xpts'], t12.data['avgq'])

### T2

In [ ]:
t2 = meas.T2Experiment(cfg_dict,qi=qi, params={'active_reset':True})
t22 = meas.T2Experiment(cfg_dict,qi=qi, params={'active_reset':False})

plt.figure()
plt.plot(t2.data['xpts'], t2.data['avgi'], label='Active Reset')
plt.plot(t22.data['xpts'], t22.data['avgi'])
plt.legend()
# plt.figure()
# plt.plot(t2.data['xpts'], t2.data['avgq'])
# plt.plot(t22.data['xpts'], t22.data['avgq'])

# histograms are different for active reset vs no active reset!!! because it's getting the full set of measured data. you should adjust to the first one. 
plt.figure()
plt.plot(t2.data['bin_centers'], t2.data['hist'])
plt.plot(t22.data['bin_centers'], t22.data['hist'])

### Echo

In [ ]:
t2 = meas.T2Experiment(cfg_dict,qi=qi, params={'active_reset':True, 'experiment_type':'echo'})
t22 = meas.T2Experiment(cfg_dict,qi=qi, params={'active_reset':False, 'experiment_type':'echo'})

plt.figure()
plt.plot(t2.data['xpts'], t2.data['avgi'])
plt.plot(t22.data['xpts'], t22.data['avgi'])

plt.figure()
plt.plot(t2.data['bin_centers'], t2.data['hist'])

# change the hist to take the first measurement 
plt.plot(t22.data['bin_centers'], t22.data['hist'])

## Sweep threshold

In [ ]:
d = []
qi=4
auto_cfg = config.load(cfg_path)
threshold = auto_cfg['device']['readout']['threshold'][qi]
rng=4*auto_cfg['device']['readout']['sigma'][qi]

thresh = np.linspace(threshold-rng/2,threshold+rng/2,12)
for i, t in enumerate(thresh):
    shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':10000,'threshold_v':t, 'active_reset':True,'setup_reset':True, 'reset':4}, display=False, progress=False)
    d.append(shot)
    if i%4==0:
        print(f'Completed {i}/{len(thresh)}')
    #shot.check_reset()


In [ ]:
import slab_qick_calib
slab_qick_calib.calib.readout_helpers.plot_reset(d,shot.fname)

## Turn off active reset for all config chans

In [ ]:
for qi in range(20):
    config.update_readout(cfg_path, 'active_reset',False, qi)

## Turn on active reset for channels where it seems to be working

In [ ]:
e_success = 0.15 # Ratio of e proportion after active reset compared to before
g_vs_e = 2 # Ratio of g proportion to e proportion after active reset

auto_cfg = config.load(cfg_path)
reset_e = auto_cfg['device']['readout']['reset_e']
reset_g = auto_cfg['device']['readout']['reset_g']
result = np.array(reset_e)< e_success | np.array(reset_g)/np.array(reset_e)<g_vs_e
for qi in range(20):
    config.update_readout(cfg_path, 'active_reset',bool(result[qi]), qi)

## Check reset (plotting result of reset)

In [ ]:
qubit_list = np.arange(3)
qubit_list =[0]
for qi in qubit_list:
    shot = meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':50000,'active_reset':True})
    shot.check_reset()

# Chi

In [ ]:
# Need a tuned up pi pulse for this
span = 30
qubit_list = [0]
#qubit_list = np.arange(6)
for qi in qubit_list: 
    #chi, chi_val=measure_func.check_chi(cfg_dict, qi)
    chi, chi_val=measure_func.check_chi(cfg_dict, qi, span=span, df=-6)
    auto_cfg = config.update_readout(cfg_path, 'chi', chi_val, qi)

# 2 Qubit

In [ ]:
t12q = meas.T1_2Q(cfg_dict, qi=[10,0], params={'active_reset':False, })

In [ ]:
rabi2q = meas.Rabi_2Q(cfg_dict, qi=[10,0], params={'active_reset':True})

# EF 

### Initial setting of frequencies based on guess for alpha

In [ ]:
# Initial set of the frequencyies based on guess for alpha 
alpha = -240
qubit_list = np.arange(2)
#qubit_list = [1]
auto_cfg = config.load(cfg_path)
for i in qubit_list: 
    f_ge = auto_cfg['device']['qubit']['f_ge'][i]
    auto_cfg = config.update_qubit(cfg_path, 'f_ef', f_ge+alpha, i)

## Spectroscopy EF

### General search

In [ ]:
bad_qubits=[]

qubit_list=[0]
qubit_list=np.arange(6)

for qi in qubit_list:
    status, ntries = qubit_tuning.find_spec(qi, cfg_dict, start='medium', freq='ef')
    if not status:
        bad_qubits.append(qi)

### Specific width

In [ ]:
# You may want to update this frequency, which will be the center of the scan. 
#style huge, coarse, medium, fine 
update=True

qubit_list = np.arange(3)
qubit_list=[1]
for qi in qubit_list:
    qspec=meas.QubitSpec(cfg_dict, qi=qi, style='medium', params={'checkEF':True,'gain':0.0001, 'span':20,'length':5,'reps':5000})
    #qspec=meas.QubitSpec(cfg_dict, qi=qi, style='coarse', params={'checkEF':True, 'reps':750,'gain':0.15, 'span':200})

    #qspec=meas.QubitSpec(cfg_dict, qi=qi, style='medium', params={'checkEF':True})#, params={'span':500, 'expts':1000,'reps':500,'gain':0.2})
    if update and qspec.status:
        auto_cfg = config.update_qubit(cfg_path, 'f_ef', qspec.data["best_fit"][2], qi)

## Rabi EF

In [ ]:
# If first time, initialize the sigma and gain to those of the ge 
first_time = False
update = True

qubit_list = np.arange(6)
qubit_list=[1]

bad_qubits = []
auto_cfg = config.load(cfg_path)

for qi in qubit_list: 
    if first_time:
        #config.update_qubit(cfg_path, 'f_ef', auto_cfg.device.qubit.f_spec_ef[qi], qi)
        auto_cfg = config.update_qubit(cfg_path, ('pulses','pi_ef','sigma'), auto_cfg['device']['qubit']['pulses']['pi_ge']['sigma'][qi], qi)
        auto_cfg = config.update_qubit(cfg_path, ('pulses','pi_ef','gain'), auto_cfg['device']['qubit']['pulses']['pi_ge']['gain'][qi], qi)
    amp_rabi = meas.RabiExperiment(cfg_dict,qi=qi, params={'checkEF':True, 'reps':400})
    if update and amp_rabi.status:
        config.update_qubit(cfg_path, ('pulses','pi_ef','gain'), amp_rabi.data['pi_length'], qi)
    else:
        print(f'Amplitude Rabi fit failed for qubit {qi}')
        bad_qubits.append(qi)

    # MAX GAIN IS GOING TO 5 

In [ ]:
qi=1
measure_temp(cfg_dict, qi, bf_client=bf_client, temp=40)

## Qubit Temperature

In [ ]:
qubit_list = np.arange(3)
qubit_list=[1]
# Setting number of rounds multiplies the default number of rounds by that number, so can be greater than or less than 1. 
# Make sure to run single shot first 
for qi in qubit_list: 
    # rounds will make scan take longer, needed for lower temperatures. 
    temp, pop,_,_ = measure_func.measure_temp(cfg_dict, qi=qi, temp=40)# , rounds=5)
    auto_cfg = config.update_qubit(cfg_path, 'temp', temp, qi)
    auto_cfg = config.update_qubit(cfg_path, 'pop', pop, qi)

## Calculate Ec and EJ

In [ ]:
auto_cfg = config.load(cfg_path)
import scqubits as scq
for i in np.arange(3):
    alpha =  auto_cfg.device.qubit.f_ef[i] - auto_cfg.device.qubit.f_ge[i]
    en = scq.Transmon.find_EJ_EC(auto_cfg.device.qubit.f_ge[i]/1000,alpha/1000)
    print(alpha)
    print(en)
    q = np.pi*2 * auto_cfg.device.qubit.f_ge[i] * auto_cfg.device.qubit.T1[i]
    print(q)

## Ramsey EF

In [ ]:
update=True
qubit_list=np.arange(3)
qubit_list=[1]

for qi in qubit_list:
    t2r = meas.T2Experiment(cfg_dict, qi=qi, params={'ramsey_freq':0.15, 'checkEF':True})

    if update and t2r.status:
        config.update_qubit(cfg_path, 'f_ef', t2r.data['new_freq'], qi)
    else:
        print('T2 Ramsey fit failed')

# Calculate qubit params

In [ ]:
# If this is the first time you're doing this, create new file 
qubit_params.ham(cfg_path)
qubit_params.delta(cfg_path)
qubit_params.cohere(cfg_path)

# Stark

## Ramsey

### Single experiment

In [ ]:
qi=0
t2_stark = meas.RamseyStarkExperiment(cfg_dict, qi=qi, params={'stark_gain':1,'df':40,'acStark':True,'ramsey_freq':0.1, 'expts':100})

### Create stark part of config file for tracking Gain-> Freq
Only do once for each config

In [ ]:
# config.init_stark_section(cfg_dict['cfg_file'], 6)

### Sweep frequency

In [ ]:
qubit_list = np.arange(3)
qubit_list=[3]

gain = np.linspace(0.1,1,10)
for qi in qubit_list:
    for g in gain:
        t2rstark=meas.RamseyStarkFreqExperiment(cfg_dict, qi=qi, params={'step':1/430+0.001, 'stark_gain':g, 'start_df':30, 'end_df':150, 'expts_df':10})

### Calibrate stark power positive freq

You'll want to adjust the step size / expts to capture the full frequency range 

In [ ]:
qubit_list = np.arange(3)
qubit_list=[0]
d=[]
freqs= [50]
update=True
for f in freqs:
    for qi in qubit_list:
        params={'step':0.015, 'expts_gain':16, 'df':f, 'start_gain':0.035, 'end_gain':0.16,'expts':140, 'ramsey_freq':0.1}
        t2rstark=meas.RamseyStarkPowerExperiment(cfg_dict, qi=qi, params=params, live_plot=True)
        d.append(t2rstark)
        if update: t2rstark.update(neg=False)

In [ ]:
plt.figure()
for i in range(len(t2rstark.data['bin_centers'])):
    plt.plot(t2rstark.data['bin_centers'][i], t2rstark.data['hist'][i])

### Negative frequency

In [ ]:
#qubit_list = np.arange(3)
update=True 
qubit_list=[0]
df = -45
for qi in qubit_list:
    params = {'step':0.015, 'expts_gain':12, 'df':df, 'start_gain':0.04, 'end_gain':0.11, 'expts':140}
    t2rstark=meas.RamseyStarkPowerExperiment(cfg_dict, qi=qi, params=params)#, live_plot=True)
    if update: t2rstark.update(neg=True)
#handy.plot_many(d, title='Ramsey Stark', save_path=cfg_dict['expt_path'])

## T1

### Single exp

In [ ]:
qi=1
gain_list = [2]
for g in gain_list:
    params = {'stark_gain':g,'df':40,'expts':60, 'span':106}
    #params={'active_reset':False, 'df':200, 'stark_gain':g,'expts':300,'start':10,'span':0,'reps':1000}
    t1 = meas.T1StarkExperiment(cfg_dict, qi=qi, params=params) 

### Gain sweep

In [ ]:
qubit_list = np.arange(3)
qubit_list=[0]

flist=[-45]
for f in flist:
    for qi in qubit_list: 
        #t1_neg = meas.T1StarkPowerExperiment(cfg_dict, qi=qi, params={'df':-70,'start_gain':0.02,'end_gain':0.2,'start':3, 'rounds':4})
        params={'df':f,'start_gain':0,'end_gain':0.14,'start':0.2, 'rounds':1,'span':103,'expts_gain':20, 'expts':80}
        t1_pos = meas.T1StarkPowerExperiment(cfg_dict, qi=qi, params=params)#, live_plot=True) 
        plt.figure()
        for i in range(len(t1_pos.data['bin_centers'])):
            plt.plot(t1_pos.data['bin_centers'][i], t1_pos.data['hist'][i])

### Freq sweep

In [ ]:
t1_freq = meas.T1StarkFreqExperiment(cfg_dict, qi=19, params={'span':6, 'span_f':100, 'start_df':50, 'expts_f':100})

### Single evo point linear gain sweep

In [ ]:
t1_cont = meas.T1StarkPowerSingle(cfg_dict, qi=0, params={"rounds":1, 'wait_time':99, 'expts':100})

### Sweep frequency linearly, 1d scan

In [ ]:
t1_quad = meas.T1StarkPowerQuadSingle(cfg_dict, qi=0, params={"rounds":1, 'wait_time':99, 'expts':300, "stop_f":3})

# It would be good to add a few excited state and ground state measurements to this. 

In [ ]:
t1_norm = -1/np.log((t1_quad_2d.data['scale_data']-shot.data['g_norm'])/shot.data['e_norm'])*tau/auto_cfg.device.qubit.T1[qi]

### Sweep frequency linearly, 2d scan sweep over time

In [ ]:
qi=0
shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':30000, 'active_reset':True,'setup_reset':False, 'reset':2})
t1 = meas.T1Experiment(cfg_dict, qi=qi)

In [ ]:
print(shot.data['g_norm'])
print(shot.data['e_norm'])

In [ ]:
print(shot.data['vg'])
print(shot.data['ve'])

### Corrected 2D scan

In [ ]:
qi = 0

#params={'wait_time':tau, "stop_f":6, 'expts':200, "sweep_pts":10, 'g_norm':shot.data['g_norm'], 'e_norm':shot.data['e_norm']}

for i in range(1):
    shot=meas.HistogramExperiment(cfg_dict, qi=qi, params={'shots':30000, 'active_reset':True,'setup_reset':False, 'reset':2})
    #shot.update()
    t1 = meas.T1Experiment(cfg_dict, qi=qi)
    t1.update()
    if t1.data['new_t1']>103:
        tau = 103
    else:
        tau = t1.data['new_t1']
    #params={"wait_time":tau, "stop_f":6, 'expts':300, "sweep_pts":15, 'g_norm':shot.data['g_norm'], 'e_norm':shot.data['e_norm'], 'vg':shot.data['vg'], 've':shot.data['ve']}
    params={"wait_time":tau, "df":6, 'expts':300, "sweep_pts":15, 'g_norm':shot.data['g_norm'], 'e_norm':shot.data['e_norm'], 'vg':shot.data['vg'], 've':shot.data['ve']}

    t1_quad_2d = meas.T1StarkPowerQuad2D(cfg_dict, qi=qi,params=params)# , live_plot=True)

In [ ]:
t1_quad_2d.data['stark_gain_pts'][0]

In [ ]:
t1_quad_2d.cfg.expt.wait_time

In [ ]:
plt.figure()
for i in range(len(t1_quad_2d.data['bin_centers'])):
    #plt.plot(t1_quad_2d.data['bin_centers'][i], t1_quad_2d.data['hist'][i])
    plt.plot(t1_quad_2d.data['f_pts'][i], t1_quad_2d.data['t1_norm'][i])#+0.43*i)

In [ ]:
self.data['t1_norm'] = -1/np.log((self.data['scale_data']-self.cfg.expt['g_norm'])/self.cfg.expt['e_norm'])*self.cfg.expt.wait_time/self.cfg.device.qubit.T1[q]


In [ ]:
qi=0

tau=104
t1_quad_2d = meas.T1StarkPowerQuad2D(cfg_dict, qi=qi, params={'wait_time':tau, "stop_f":3.5, 'expts':10, "sweep_pts":100}, live_plot=True)

In [ ]:
plt.figure()
for i in range(len(t1_quad_2d.data['bin_centers'])):
    plt.plot(t1_quad_2d.data['bin_centers'][i], t1_quad_2d.data['hist'][i])

In [ ]:
plt.figure()
plt.imshow(t1_norm, aspect='auto', origin='lower')
plt.colorbar(label='avgi')
plt.xlabel('Sweep Points')
plt.ylabel('Wait Time Index')

In [ ]:
len(t1_quad_2d.data['xpts'])

In [ ]:
t1_quad_2d.data['xpts']

In [ ]:
t1_quad_2d.display()